# Transcriptomics Agentic Web Platform

### **[대장암 paired bulk RNA-seq 데이터를 이용한 전사체 분석 Agent와 Agentic Web Platform을 구축]**

- **기본 구성**
  - 데이터 스크리닝 (raw count matrix, metadata 스크리닝)
  - DEG 분석 및 시각화
  - Enrichment Analysis
- **웹 플랫폼 구축**
  - 분석 Agent를 Gradio에 연결하여 대화형 분석 플랫폼 구축

<br>

분석에는 아래의 **raw count와 metadata 두 파일**을 사용함

- `/content/drive/MyDrive/2026-BIML-Jeon-Agentic_AI/data/GSE95132_colorectal_cancer_paired_tumor_vs_adjacent_normal_raw_counts.tsv`
- `/content/drive/MyDrive/2026-BIML-Jeon-Agentic_AI/data/GSE95132_colorectal_cancer_paired_tumor_vs_adjacent_normal_metadata.tsv`

[참고] Gemini API key는 Colab Secrets에 `GOOGLE_API_KEY`라는 이름으로 저장할 것


---

## 실습 데이터 출처와 구성

- 본 실습은 NCBI Gene Expression Omnibus(GEO)에 공개된 human colorectal cancer RNA-seq 자료 **[GSE95132](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE95132)**를 사용함
  - 전체 series는 KRAS-mutant colorectal tumor와 환자별 adjacent normal colon tissue 10쌍, 그리고 aberrant crypt foci(ACF)와 normal crypt 5쌍으로 구성되며 Illumina HiSeq 2500으로 측정됨
- 실습에는 **대장암 환자 10명의 tumor–adjacent normal paired sample 20개만 사용**함 (ACF/normal crypt sample과 technical replicate는 포함하지 않음)

| 항목 | 내용 |
|---|---|
| Organism | *Homo sapiens* |
| 표본 | Primary tumor 10개 + matched adjacent normal 10개 |
| 비교 방향 | tumor vs. normal |
| Platform | Illumina HiSeq 2500 (GPL16791) |
| Reference genome | hg19 |

- `raw_counts.tsv`는 FASTQ 원자료가 아니라 PyDESeq2 입력에 사용하는 **23,640 genes × 20 samples의 gene-level count matrix** 임
- `metadata.tsv`에는 sample, patient, condition 및 원본 sample 정보가 들어 있음
- 이들은 실습을 위해 한 번 더 정리한 파일임

<br>

## 원 논문과의 관계

- 이 자료는 Hanley 등의 논문 “Genome-wide DNA methylation profiling reveals cancer-associated changes within early colonic neoplasia”에 연결된 RNA-seq 자료임
- 원 연구는 초기 대장 종양성 병변인 ACF와 colorectal cancer에서 발생하는 DNA methylation 변화를 조사했으며, RNA-seq는 methylation과 gene expression의 관계를 평가하기 위해 함께 사용됨
- 논문에서는 ACF에서 확인된 methylation 변화의 상당수가 colorectal cancer에서도 관찰되었고, colorectal cancer에서는 promoter methylation과 gene expression 변화 사이의 연관성이 보고됨
- 다만, 본 실습은 원 논문의 methylation 분석을 재현하는 것이 아니라 같은 연구에서 공개한 paired RNA-seq subset으로 **tumor–normal DEG와 pathway를 분석하는 교육용 workflow**임

> **해석 시 유의점**
> - adjacent normal은 별도의 건강한 사람에게서 얻은 control이 아니라 암 환자의 종양 인접 조직임
> - 표본이 10 patient pair이고 KRAS-mutant tumor 중심이므로 결과를 모든 대장암 환자에게 일반화하기 어려움

### 참고 자료

- [NCBI GEO: GSE95132](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE95132)
- [Hanley MP et al., *Oncogene* (2017)](https://pubmed.ncbi.nlm.nih.gov/28459462/)
- DOI: [10.1038/onc.2017.130](https://doi.org/10.1038/onc.2017.130)


## 0. Colab 실행 환경 준비

### 필수 패키지 설치

In [ ]:
!pip install -q --upgrade-strategy only-if-needed \
    "numpy==2.0.2" \
    "pandas==2.2.2" \
    "requests==2.32.5" \
    "deepagents==0.6.12" \
    "langchain==1.3.14" \
    "langchain-core==1.5.1" \
    "langchain-google-genai==4.3.2" \
    "langchain-community==0.4.1" \
    "langchain-huggingface==1.2.2" \
    "langchain-text-splitters==1.1.2" \
    "langgraph==1.2.9" \
    "langgraph-checkpoint==4.1.1" \
    "gradio==6.14.0" \
    "pydeseq2==0.5.4" \
    "gseapy==1.3.1" \
    "faiss-cpu==1.12.0" \
    "sentence-transformers==5.7.0" \
    "python-dotenv==1.2.2"

In [ ]:
# ============================================================
# 기본 데이터 처리 및 시각화
# ============================================================

import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt


# ============================================================
# 기본 Python 및 환경 설정
# ============================================================

import ast
import json
import queue
import re
import threading
import time
import zipfile
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Image, display


# ============================================================
# LangChain 기본 구성
# ============================================================

from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


# ============================================================
# LangGraph Memory
# ============================================================

from langgraph.checkpoint.memory import InMemorySaver


# ============================================================
# 전사체 분석 및 Enrichment Analysis
# ============================================================

import gseapy as gp
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


# ============================================================
# GSEApy 문서 기반 RAG
# ============================================================

import faiss
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


# ============================================================
# Deep Agent 및 Backend
# ============================================================

from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend


# ============================================================
# Gradio Web UI
# ============================================================

import gradio as gr

### Google Drive 및 Gemini API key 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("Google Drive 및 Gemini API key 연결 완료")


## 실습 데이터와 통계 설계

- 입력 행렬: gene × sample raw integer count
- 표본: 10명 환자의 tumor 및 adjacent normal, 총 20개 표본
- 설계식: `~patient + condition`
- contrast: `tumor vs normal`
- LFC 방향: `log2(tumor / normal)`

동일 환자의 두 조직이 서로 독립적이지 않으므로 `patient` 효과를 설계식에 포함함. 전체 통계표와 필터링된 DEG 결과를 별도로 보존함

## 공통 Python 환경과 경로

In [ ]:
import ast
import json
import queue
import re
import threading
import time
from pathlib import Path

from IPython.display import Image, display


PROJECT_DIR = Path(
    "/content/drive/MyDrive/2026-BIML-Jeon-Agentic_AI"
)
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/BIML-Jeon-Output"
)
KNOWLEDGE_BASE_DIR = (
    PROJECT_DIR / "knowledge_base" / "gseapy"
)


# ============================================================
# Input
# ============================================================

COUNT_PATH = DATA_DIR / (
    "GSE95132_colorectal_cancer_"
    "paired_tumor_vs_adjacent_normal_raw_counts.tsv"
)

METADATA_PATH = DATA_DIR / (
    "GSE95132_colorectal_cancer_"
    "paired_tumor_vs_adjacent_normal_metadata.tsv"
)

GSEAPY_DOCUMENT_PATHS = [
    KNOWLEDGE_BASE_DIR / "gseapy_intro.txt",
    KNOWLEDGE_BASE_DIR / "gseapy_run.txt",
]


# ============================================================
# Output
# ============================================================

STATS_PATH = OUTPUT_DIR / (
    "GSE95132_CRC_paired_tumor_vs_normal_full_stats.csv"
)

DEG_PATH = OUTPUT_DIR / (
    "GSE95132_CRC_paired_tumor_vs_normal_DEGs.csv"
)

VOLCANO_PATH = OUTPUT_DIR / (
    "GSE95132_CRC_paired_tumor_vs_normal_volcano.png"
)


# ============================================================
# 필수 입력 파일 확인
# ============================================================

for path in [
    COUNT_PATH,
    METADATA_PATH,
    *GSEAPY_DOCUMENT_PATHS,
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"필수 입력 파일을 찾을 수 없음: {path}"
        )


print(f"Count matrix : {COUNT_PATH}")
print(f"Metadata     : {METADATA_PATH}")
print(f"GSEApy docs  : {KNOWLEDGE_BASE_DIR}")
print(f"Outputs   : {OUTPUT_DIR}")

# Part 0. LLM과 대화하기

## Chat Model 만들기

`ChatGoogleGenerativeAI`는 Gemini API를 LangChain의 공통 Chat Model interface로 감싼 객체임
- 링크: https://reference.langchain.com/python/langchain-google-genai/chat_models/ChatGoogleGenerativeAI

이후 `create_agent()`와 `create_deep_agent()`가 동일 객체를 사용하며 이를 이용하여 LLM과 대화 가능함

`InMemoryRateLimiter`는 호출 간격을 제한하는 방식임
- 우리는 무료 API를 사용하고 있기 때문에 제한에 걸리지 않기 위해서 사용함
- 아래 설정은 약 5초마다 한 번 호출하여 Flash-Lite의 분당 제한 요청수인 15 RPM보다 여유를 둠
- 모든 Main/Subagent가 **동일 limiter 객체**를 공유해야 합산 호출 속도가 제어됨

In [ ]:
## 기본적인 모델 호출 방식은 아래와 같음
## 우리는 무료 API 사용으로 인해 중간에 모델을 변경할 수도 있기 때문에 따로 함수를 구현함

# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",
#                             api_key="...")

In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL_NAME = "gemini-3.5-flash-lite"

shared_rate_limiter = InMemoryRateLimiter(
    requests_per_second=1 / 5,
    check_every_n_seconds=0.5,
    max_bucket_size=1,
)

def make_llm():
    return ChatGoogleGenerativeAI(
        model=MODEL_NAME,
        rate_limiter=shared_rate_limiter,
    )

llm = make_llm()
print(f"Model: {MODEL_NAME}")

In [ ]:
## 참고용 (다른 모델을 사용할 경우)

# OpenAI
!pip install -q langchain-openai
from langchain_openai import ChatOpenAI

ChatOpenAI(model="gpt-5-mini")

# Anthropic Claude
!pip install -q langchain-anthropic
from langchain_anthropic import ChatAnthropic

ChatAnthropic(model="claude-sonnet-4-5")

# Mistral
!pip install -q langchain-mistralai
from langchain_mistralai import ChatMistralAI

ChatMistralAI(model="mistral-large-latest")

## LangChain 객체 실행법 & 실시간 Logger 구축

### LangChain Message 객체

LangChain은 실행 기록을 Message 객체로 구분함

- `AIMessage`: 모델의 답변 또는 Tool 호출 요청
- `ToolMessage`: Tool 실행 결과

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

- Chat Model과 Agent는 이러한 Message 객체를 입력과 출력으로 사용함
- 실행할 때는 `invoke()` 또는 `stream()` method를 사용함

---
### invoke()

- invoke(input)는 입력 하나를 전달하고 전체 작업이 완료될 때까지 기다린 뒤 최종 결과를 반환하는 동기 실행 method임
- 실행이 완료되기 전까지는 중간 Tool 호출이나 진행 상태가 Notebook에 표시되지 않음

```python
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "데이터를 점검해줘.",
        }
    ]
})
```

동작 과정은 다음과 같음

> 입력 전달
→ 모델 판단
→ Tool 호출
→ Tool 결과 확인
→ 후속 Tool 호출
→ 최종 답변 생성
→ 전체 결과 반환

작업이 끝난 뒤에는 전체 message history를 확인할 수 있음

``` python
result["messages"]
```


---
### stream()

- stream(input)은 작업 전체가 끝날 때까지 기다리지 않고, 실행 중 발생하는 결과를 순차적으로 전달하는
method임
- stream_mode="updates"는 Agent graph의 node가 state를 갱신할 때마다 해당 변경 내용을 전달하도록 지정함

```python
for update in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "데이터를 점검해줘.",
            }
        ]
    },
    stream_mode="updates",
):
    print(update)
```


In [ ]:
### `invoke()` 사용

messages = [
    (
        "system",
        "당신은 생명정보학 강의를 돕는 친절한 AI 조교입니다.",
    ),
    (
        "human",
        "한 문장으로 인사하고, 오늘 배울 내용을 간단히 소개해줘.",
    ),
]

response = llm.invoke(messages)

print(response.content)

# system 메시지는 모델의 역할과 답변 방식을 설정함.
# human 메시지는 사용자의 실제 질문이나 요청을 전달함.
# invoke()는 완성된 응답을 AIMessage 객체로 반환하며 생성된 답변은 response.content에서 확인할 수 있음.


In [ ]:
### `stream()` 사용

messages = [
    (
        "system",
        "당신은 생명정보학 강의를 돕는 친절한 AI 조교입니다.",
    ),
    (
        "human",
        "자기소개와 전사체 분석에서 할 수 있는 일을 "
        "각각 한 문장씩 설명해줘.",
    ),
]

for chunk in llm.stream(messages):
    print(chunk.text, end="", flush=True)

# stream()은 응답 전체를 한 번에 반환하지 않고 생성되는 내용을 AIMessageChunk 단위로 전달함.
# 각 chunk의 text를 이어서 출력하면 모델의 답변이 생성되는 과정을 실시간으로 확인할 수 있음.

> **본 실습에서는 길이가 긴 Prompt, Query를 사용하기 때문에 Agent의 동작 과정을 실시간으로 확인할 수 있는 Logger를 만듦**

**아래 logger가 표시하는 항목은 다음과 같음**

- Todo 상태 변경
- Tool/Subagent 호출과 완료
- 30초 동안 이벤트가 없을 때 진행 표시
- 최종 text 답변
- Tool observation에서 확인된 실제 이미지
- API 및 파일 오류의 짧은 한국어 안내

In [ ]:
# 실행 대상에 따라 두 Logger를 사용함.
# - run_model_with_log(): Chat Model의 생성 text를 실시간 출력함.
# - run_with_compact_log(): Main/Subagent의 작업, Todo, Tool 및 최종 답변을 출력함.

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}


def text_only(content):
    """문자열 또는 provider content block에서 사용자용 text만 추출함."""
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "\n".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict)
            and block.get("type") == "text"
        ).strip()

    return str(content)


def _short(value, limit=240):
    """로그 값을 한 줄의 제한된 길이로 변환함."""
    text = (
        value
        if isinstance(value, str)
        else json.dumps(
            value,
            ensure_ascii=False,
            default=str,
        )
    )
    text = " ".join(text.split())

    return (
        text
        if len(text) <= limit
        else text[: limit - 1] + "…"
    )


def _normalize_todos(value):
    """Tool argument에 문자열 또는 list로 담긴 Todo를 list로 정규화함."""
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            try:
                value = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return []

    return value if isinstance(value, list) else []


def _friendly_agent_name(name):
    """내부 Agent 이름을 읽기 쉬운 표시 이름으로 변환함."""
    known_names = {
        "transcriptomics_analysis_agent": "전사체 분석 Agent",
        "gseapy_documentation_agent": "GSEApy 문서 Agent",
        "crc_deep_multi_agent": "Main Agent",
        "crc_filesystem_deep_agent": "파일 분석 Agent",
    }

    if not name:
        return "Subagent"
    if name in known_names:
        return known_names[name]
    return str(name).replace("_", " ").strip().title()


def _friendly_error(exc):
    """파일 및 Gemini quota 오류를 가능한 범위에서 구분해 설명함."""
    text = str(exc)
    lower = text.lower()

    if (
        "requestsperday" in lower
        or "requests_per_day" in lower
        or "perday" in lower
    ):
        return (
            "Gemini 일일 요청 한도(RPD)에 도달함. "
            "새 thread로 바꾸어도 quota는 초기화되지 않음."
        )

    if (
        "requestsperminute" in lower
        or "requests_per_minute" in lower
        or "perminute" in lower
    ):
        return (
            "Gemini 분당 요청 한도(RPM)에 도달함. "
            "현재 thread를 유지한 채 잠시 기다린 후 재시도 가능함."
        )

    if (
        "tokensperminute" in lower
        or "tokens_per_minute" in lower
    ):
        return (
            "Gemini 분당 token 한도(TPM)에 도달함. "
            "입력 context를 줄이거나 잠시 기다린 후 재시도 필요함."
        )

    if "resource_exhausted" in lower or "429" in lower:
        return (
            "Gemini API가 RESOURCE_EXHAUSTED(429)를 반환함. "
            "원본 응답에서 RPM/RPD/TPM quota 항목 확인 필요함. "
            f"원본: {_short(text, 300)}"
        )

    if isinstance(exc, FileNotFoundError):
        return f"필요한 파일을 찾지 못함: {exc.filename or text}"

    return (
        f"작업 중단: {type(exc).__name__}: "
        f"{_short(text, 300)}"
    )


def _snapshot_output_images():
    """
    현재 OUTPUT_DIR 내부 이미지의 version 정보를 반환함.

    이미지를 표시하는 함수가 아니라 요청 전후의 생성·수정 여부를
    비교하기 위한 snapshot 함수임.
    """
    output_root = OUTPUT_DIR.resolve()
    snapshot = {}

    if not output_root.is_dir():
        return snapshot

    for path in output_root.rglob("*"):
        if (
            not path.is_file()
            or path.suffix.lower() not in IMAGE_EXTENSIONS
        ):
            continue

        try:
            resolved = path.resolve()
            stat = resolved.stat()
        except (OSError, RuntimeError):
            continue

        if not resolved.is_relative_to(output_root):
            continue

        # 동일 경로를 덮어쓴 경우에도 수정 여부를 탐지함.
        snapshot[resolved] = (
            stat.st_mtime_ns,
            stat.st_size,
        )

    return snapshot


def run_model_with_log(
    model,
    messages,
    *,
    label="LLM 호출",
):
    """단순 Chat Model의 streaming text를 실시간 출력함."""
    print(f"[START] {label}", flush=True)
    parts = []

    try:
        for chunk in model.stream(messages):
            text = text_only(chunk.content)

            if text:
                print(text, end="", flush=True)
                parts.append(text)

    except Exception as exc:
        print(f"\n[오류] {_friendly_error(exc)}", flush=True)
        return None

    print(f"\n[END] {label}", flush=True)
    return "".join(parts)


def run_with_compact_log(
    agent,
    query,
    thread_id,
    *,
    subgraphs=False,
    idle_notice_seconds=30,
):
    """
    Agent 실행 과정을 사람이 읽기 쉬운 계층형 로그로 출력함.

    이미지 표시 원칙
    - 요청 시작 전에 OUTPUT_DIR의 기존 이미지 상태를 기록함.
    - 현재 요청 중 새로 생성되거나 수정된 이미지만 수집함.
    - 과거 요청에서 생성됐고 이번 요청에서 변경되지 않은 이미지는 표시하지 않음.
    - 같은 경로를 이번 요청에서 덮어쓴 경우에도 변경된 이미지로 인식함.
    - glob/read_file로 기존 이미지 경로를 발견한 것만으로는 표시하지 않음.
    - 수집한 이미지는 Main Agent의 최종 답변을 출력한 뒤 한 번에 표시함.
    """
    started = time.monotonic()
    final = None

    seen_messages = set()
    seen_tool_calls = set()
    active_calls = {}

    pending_subagents = queue.SimpleQueue()
    namespace_labels = {}

    current_agent = "Main Agent"
    current_action = "응답 대기"

    # 요청이 시작되기 전에 이미 존재하던 이미지 상태임.
    image_snapshot = _snapshot_output_images()

    # 현재 요청에서 생성·수정된 이미지와 version을 모아둠.
    collected_image_paths = set()
    collected_image_versions = set()

    compact_read_tools = {
        "glob",
        "grep",
        "read_file",
        "ls",
        "search_gseapy_documentation",
    }

    important_tools = {
        "task",
        "execute",
        "write_file",
        "edit_file",
        "summarize_selected_crc_results",
        "run_selected_crc_deseq2",
    }

    def elapsed_seconds():
        return int(time.monotonic() - started)

    def stamp():
        elapsed = elapsed_seconds()
        return f"[{elapsed // 60:02d}:{elapsed % 60:02d}]"

    def duration(since):
        seconds = max(0, int(time.monotonic() - since))
        if seconds >= 60:
            return f"{seconds // 60}분 {seconds % 60}초"
        return f"{seconds}초"

    def namespace_key(namespace):
        if not namespace:
            return "main"
        return " > ".join(str(part) for part in namespace)

    def scope_for(namespace):
        """처음 관찰된 subgraph namespace에 대기 중인 task 이름을 연결함."""
        nonlocal current_agent

        key = namespace_key(namespace)
        if key == "main":
            return "Main Agent"

        if key not in namespace_labels:
            try:
                task_info = pending_subagents.get_nowait()
            except queue.Empty:
                task_info = None

            if task_info:
                namespace_labels[key] = task_info["label"]
            else:
                namespace_labels[key] = f"Subagent {len(namespace_labels) + 1}"

        current_agent = namespace_labels[key]
        return current_agent

    def result_count(content):
        value = content

        if isinstance(value, str):
            try:
                value = json.loads(value)
            except json.JSONDecodeError:
                try:
                    value = ast.literal_eval(value)
                except (ValueError, SyntaxError):
                    value = None

        if isinstance(value, (list, tuple, set)):
            return f"{len(value)}개"
        if isinstance(value, dict):
            return f"{len(value)}개 항목"
        return _short(content, 180)

    def collect_changed_images():
        """직전 확인 이후 생성·수정된 이미지를 최종 표시 목록에 수집함."""
        nonlocal image_snapshot

        current_snapshot = _snapshot_output_images()
        changed_images = [
            (path, version)
            for path, version in current_snapshot.items()
            if image_snapshot.get(path) != version
        ]

        for image_path, version in changed_images:
            image_version = (image_path, version)

            if image_version in collected_image_versions:
                continue

            collected_image_paths.add(image_path)
            collected_image_versions.add(image_version)

        # 다음 Tool 완료 시 새 변경분만 찾도록 기준 상태를 갱신함.
        image_snapshot = current_snapshot

    def display_collected_images():
        """현재 요청에서 수집한 이미지를 최종 답변 뒤에 표시함."""
        for image_path in sorted(collected_image_paths, key=str):
            if not image_path.is_file():
                continue

            print("\n[이번 작업에서 생성·수정된 시각화]", flush=True)
            print(image_path, flush=True)
            display(Image(filename=str(image_path)))

    print("[00:00] Main Agent 시작", flush=True)
    print(f"         thread: {thread_id}", flush=True)

    stream = agent.stream(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query,
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": thread_id,
            }
        },
        stream_mode="updates",
        subgraphs=subgraphs,
    )

    events = queue.Queue()

    def consume():
        try:
            for item in stream:
                events.put(("event", item))
        except Exception as exc:
            events.put(("error", exc))
        finally:
            events.put(("done", None))

    threading.Thread(
        target=consume,
        daemon=True,
    ).start()

    while True:
        try:
            kind, event = events.get(timeout=idle_notice_seconds)
        except queue.Empty:
            print(
                f"{stamp()} {current_agent} 진행 중"
                f" · {current_action}"
                f" · {idle_notice_seconds}초간 새 이벤트 없음",
                flush=True,
            )
            continue

        if kind == "done":
            break

        if kind == "error":
            # 오류 직전에 파일 생성이 완료됐을 가능성도 확인함.
            collect_changed_images()
            print(f"\n{stamp()} 전체 작업 실패", flush=True)
            print(f"         현재 Agent: {current_agent}", flush=True)
            print(f"         마지막 작업: {current_action}", flush=True)
            print(f"         원인: {_friendly_error(event)}", flush=True)
            display_collected_images()
            return None

        if subgraphs and isinstance(event, tuple):
            namespace, update = event
        else:
            namespace, update = (), event

        scope = scope_for(namespace)

        for _, payload in update.items():
            messages = (
                payload.get("messages", [])
                if isinstance(payload, dict)
                else []
            )

            for message in messages:
                message_id = getattr(message, "id", None)

                if isinstance(message, AIMessage) and message.tool_calls:
                    if message_id and message_id in seen_messages:
                        continue
                    if message_id:
                        seen_messages.add(message_id)

                    for call in message.tool_calls:
                        call_id = call.get("id") or repr(call)
                        if call_id in seen_tool_calls:
                            continue
                        seen_tool_calls.add(call_id)

                        name = call.get("name", "unknown_tool")
                        args = call.get("args", {})
                        call_started = time.monotonic()

                        active_calls[call_id] = {
                            "name": name,
                            "args": args,
                            "scope": scope,
                            "started": call_started,
                        }

                        if name == "write_todos":
                            current_action = "Todo 계획 갱신"
                            print(f"{stamp()} {scope} 계획", flush=True)

                            raw = (
                                args.get("todos", [])
                                if isinstance(args, dict)
                                else []
                            )
                            symbols = {
                                "completed": "✓",
                                "in_progress": "▶",
                                "pending": "○",
                            }

                            for todo in _normalize_todos(raw):
                                if isinstance(todo, dict):
                                    status = str(todo.get("status", "unknown"))
                                    print(
                                        "         "
                                        f"{symbols.get(status, '?')} "
                                        f"{todo.get('content', '')}",
                                        flush=True,
                                    )
                            continue

                        if name == "task":
                            subagent_type = (
                                args.get("subagent_type")
                                if isinstance(args, dict)
                                else None
                            )
                            description = (
                                args.get("description", "위임 작업")
                                if isinstance(args, dict)
                                else "위임 작업"
                            )
                            label = _friendly_agent_name(subagent_type)
                            active_calls[call_id]["agent_label"] = label
                            pending_subagents.put({"label": label})

                            current_agent = label
                            current_action = description
                            print(f"\n{stamp()} ┌ {label} 시작", flush=True)
                            print(
                                f"         └ 작업: {_short(description, 300)}",
                                flush=True,
                            )
                            continue

                        current_action = f"{name} 실행"

                        # 읽기 Tool은 완료될 때 호출과 결과를 한 줄로 출력함.
                        if name in compact_read_tools:
                            continue

                        if name in important_tools:
                            print(
                                f"{stamp()} {scope} · {name} 시작"
                                f" · {_short(args, 180)}",
                                flush=True,
                            )
                        else:
                            print(
                                f"{stamp()} {scope} · {name}"
                                f" · {_short(args, 180)}",
                                flush=True,
                            )

                elif isinstance(message, ToolMessage):
                    tool_call_id = getattr(message, "tool_call_id", None)
                    message_key = message_id or (
                        tool_call_id,
                        message.name,
                        _short(message.content, 120),
                    )
                    if message_key in seen_messages:
                        continue
                    seen_messages.add(message_key)

                    info = active_calls.pop(tool_call_id, None)
                    name = message.name or (info or {}).get("name", "tool")
                    tool_scope = (info or {}).get("scope", scope)
                    tool_args = (info or {}).get("args", {})
                    tool_started = (info or {}).get(
                        "started",
                        time.monotonic(),
                    )

                    if name == "write_todos":
                        pass

                    elif name == "task":
                        completed_agent = (info or {}).get(
                            "agent_label",
                            "Subagent",
                        )
                        print(
                            f"{stamp()} └ {completed_agent} 완료"
                            f" · {duration(tool_started)}",
                            flush=True,
                        )
                        current_agent = "Main Agent"
                        current_action = "Subagent 결과 정리"

                    elif name in compact_read_tools:
                        print(
                            f"{stamp()} {tool_scope} · {name} "
                            f"{_short(tool_args, 110)}"
                            f" → {result_count(message.content)}",
                            flush=True,
                        )
                        current_action = f"{name} 결과 확인"

                    else:
                        print(
                            f"{stamp()} {tool_scope} · {name} 완료"
                            f" · {duration(tool_started)}"
                            f" · {_short(message.content, 200)}",
                            flush=True,
                        )
                        current_action = f"{name} 완료"

                    # Tool 종류나 stdout 문구에 의존하지 않고 실제 파일 상태를 비교함.
                    # 현재 요청에서 생성·수정된 이미지만 최종 표시 목록에 수집함.
                    collect_changed_images()

                elif isinstance(message, AIMessage) and message.content:
                    if message_id and message_id in seen_messages:
                        continue
                    if message_id:
                        seen_messages.add(message_id)

                    # Subagent 중간 답변이 Main Agent 최종 답변을 덮지 않게 함.
                    if namespace_key(namespace) == "main":
                        candidate = text_only(message.content)
                        if candidate:
                            final = candidate

    # 마지막 ToolMessage 이후 비동기 파일 저장이 끝난 경우까지 한 번 더 확인함.
    collect_changed_images()

    print(
        f"\n{stamp()} Main Agent 종료 · 총 {duration(started)}",
        flush=True,
    )

    if final:
        print("\n[최종 답변]\n" + final, flush=True)

    # 최종 답변 다음에 이번 요청에서 생성·수정된 이미지만 표시함.
    display_collected_images()

    return final

## 간단하게 LLM을 호출하여 대화해보기

Chat Model을 호출할 때는 대화 내용을 **message 목록**으로 전달함

각 message는 작성자를 나타내는 `role`과 실제 내용인 `content`로 구성됨

| role | 의미 |
|---|---|
| `system` | 모델의 역할과 대화 전반에서 따를 행동 원칙 |
| `user` | 사용자가 모델에 전달하는 요청이나 질문 |
| `assistant` | 이전에 모델이 생성한 답변 |

예를 들어 다음과 같이 system 지시와 사용자 질문을 함께 전달할 수 있음

```python
messages = [
    {
        "role": "system",
        "content": "당신은 전사체 분석 전문가입니다.",
    },
    {
        "role": "user",
        "content": "paired RNA-seq 데이터의 특징을 설명해주세요.",
    },
]
```

`role`과 `content`는 Chat Model이 대화의 참여자와 순서를 구분하기 위한 **입력 형식**이며, 앞에서 작성한 Logger와는 별개의 개념임

- `messages`: 모델에 전달할 대화 내용
- `llm`: 응답을 생성하는 Chat Model
- `run_model_with_log()`: 모델이 생성하는 응답을 실시간으로 출력하는 사용자 정의 Logger 함수

```python
result = run_model_with_log(
    llm,
    messages,
)
```

즉, Logger가 `role`과 `content`를 정의하는 것이 아니라, message 목록을 Chat Model에 전달하고 생성되는 응답을 화면에 보여주는 역할을 함


In [ ]:
hello_text = run_model_with_log(llm, [
    ("user", "안녕? 너를 한 문장으로 소개해줘."),
], label="기본 LLM 호출")

## System Prompt 추가해보기

- 보편적으로 System Prompt는 한 질문에 대한 내용이 아니라 Agent가 계속 따라야 할 역할·근거·금지사항을 정의함


In [ ]:
SYSTEM_PROMPT = """
당신은 human paired bulk RNA-seq 분석을 담당하는 연구원입니다.

다음 원칙을 따르세요.
1. Tool 실행 결과가 없으면 실제 계산을 수행했다고 말하지 마세요.
2. 사용 예제 (GSE95132) 데이터에서 log2 fold change는 log2(tumor / normal)로 해석하세요.
3. 전체 통계 결과와 필터링된 DEG 결과를 별도로 보존하세요.
4. 답변은 한국어로 간결하게 작성하고, 실제 생성 파일과 통계 설계식을 정확히 보고하세요.
5. 마크다운 형태로 답변하지마세요.
""".strip()

In [ ]:
## System Prompt는 사용자 질의와 다르게 system 파트에 연결

prompt_text = run_model_with_log(llm, [
    ("system", SYSTEM_PROMPT),
    ("user", "사람의 paired RNA-seq에서 patient를 design에 포함하는 이유를 두 문장으로 설명해줘."),
], label="System Prompt 적용")

# Part 1. `create_agent()` 함수로 전사체 분석 Agent 만들기

- `create_agent()`는 Chat Model에 Tool과 행동 지침을 연결하여 Tool-calling Agent를 생성하는 LangChain 함수임
- 생성된 Agent는 Model–Tools 반복 loop를 통해 작업을 수행함

<br>

- 행동 방식
  - 모델이 User message를 해석
  - Tool 호출 여부와 인자를 결정
  - Tool observation을 확인한 뒤 다음 행동 또는 최종 답변을 생성

<br>

| 인자 | 역할 |
|---|---|
| `model` | 다음 행동을 판단하는 Chat Model |
| `tools` | Agent가 실행할 수 있는 기능 목록 |
| `system_prompt` | 역할과 지속 행동 원칙 |
| `name` | graph 식별자 |

## `@tool`: LLM에 실행 능력 부여

지금까지 대화한 LLM은 자연어를 생성하지만 로컬 파일이나 PyDESeq2 등의 도구를 스스로 실행/활용하지 못함

> 이때, `@tool`은 Python 함수의 이름·docstring·type hint를 schema로 변환하여 모델이 호출할 수 있게 함

---

- **`@tool`은 일반 Python 함수를 LangChain Tool로 변환하는 decorator**
  - [참고] Decorator는 함수 정의 위에 `@이름` 형태로 작성하여 기존 함수에 기능을 추가하는 Python 문법임
- LangChain의 `@tool`은 함수의 이름, docstring, parameter 이름과 type hint를 Tool schema로 변환함
- 모델은 이 schema를 참고하여 어떤 Tool을 사용할지와 어떤 인자를 전달할지 결정함

```python
# 예시

@tool
def inspect_selected_crc_data() -> dict:
    """고정된 GSE95132 paired 데이터를 점검함."""
    ...
```

- `@tool` 로 지정한 함수에서는 다음 정보가 모델에 전달됨
  - 함수 이름: 사용할 기능의 식별자
  - docstring: Tool의 목적과 사용 시점
  - parameter와 type hint: 모델이 생성해야 하는 입력 형식
  - 반환값: Tool 실행 후 Agent가 받는 observation
- 실제 계산은 Python 함수 본문이 수행하며, @tool은 해당 함수를 Agent가 선택하고 호출할 수 있도록 등록 가능한 형태로 변환하는 역할임 (@tool이 직접 분석을 수행하는 것은 아님)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from langchain_core.tools import tool
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


def _read_table(path: Path, *, index_col=None) -> pd.DataFrame:
    """확장자에 따라 TSV 또는 CSV를 읽음."""
    separator = "," if path.suffix.lower() == ".csv" else "	"
    return pd.read_csv(path, sep=separator, index_col=index_col)


def _resolve_input_path(value: str | None, default: Path) -> Path:
    """기본 데이터 또는 Gradio upload 디렉터리의 입력만 허용함."""
    path = Path(value).expanduser().resolve() if value else default.resolve()
    allowed_roots = [DATA_DIR.resolve(), (OUTPUT_DIR / "uploads").resolve()]
    if not any(path.is_relative_to(root) for root in allowed_roots):
        raise ValueError(f"허용되지 않은 입력 경로: {path}")
    if not path.is_file() or path.suffix.lower() not in {".tsv", ".csv"}:
        raise FileNotFoundError(f"유효한 TSV/CSV 입력 파일이 아님: {path}")
    return path


def _resolve_result_dir(value: str | None) -> Path:
    """분석 결과 디렉터리가 OUTPUT_DIR 내부인지 확인함."""
    output_root = OUTPUT_DIR.resolve()
    result_dir = Path(value).expanduser().resolve() if value else output_root
    if not result_dir.is_relative_to(output_root):
        raise ValueError(f"OUTPUT_DIR 밖에는 결과를 저장할 수 없음: {result_dir}")
    result_dir.mkdir(parents=True, exist_ok=True)
    return result_dir


def _result_paths(result_dir: Path) -> tuple[Path, Path, Path]:
    return (
        result_dir / STATS_PATH.name,
        result_dir / DEG_PATH.name,
        result_dir / VOLCANO_PATH.name,
    )


def _load_and_validate_crc_data(
    count_path: str | None = None,
    metadata_path: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, Path, Path]:
    """기본 또는 업로드된 count/metadata를 읽고 paired 구조를 검증함."""
    count_file = _resolve_input_path(count_path, COUNT_PATH)
    metadata_file = _resolve_input_path(metadata_path, METADATA_PATH)
    counts = _read_table(count_file, index_col=0)
    metadata = _read_table(metadata_file)

    required = {"sample", "patient", "condition"}
    if not required.issubset(metadata.columns):
        raise ValueError(f"metadata에 다음 column이 필요함: {sorted(required)}")
    if metadata["sample"].duplicated().any():
        raise ValueError("metadata에 중복 sample이 존재함.")
    if counts.index.duplicated().any():
        raise ValueError("count matrix에 중복 gene이 존재함.")
    if list(counts.columns) != metadata["sample"].tolist():
        raise ValueError("count column과 metadata sample 순서가 일치하지 않음.")
    if set(metadata["condition"]) != {"normal", "tumor"}:
        raise ValueError("condition은 normal과 tumor로 구성되어야 함.")

    paired = metadata.groupby("patient")["condition"].agg(lambda x: set(x))
    if not paired.map(lambda x: x == {"normal", "tumor"}).all():
        raise ValueError("모든 환자에게 normal과 tumor 표본이 모두 필요함.")

    values = counts.to_numpy()
    if (
        pd.isna(values).any()
        or (values < 0).any()
        or not np.equal(values, np.floor(values)).all()
    ):
        raise ValueError("count는 결측치가 없는 0 이상의 정수여야 함.")

    return counts, metadata, count_file, metadata_file


# ============================================================
# Tool 1. 기본 또는 업로드된 데이터 구조와 paired 여부 점검
# ============================================================
@tool
def inspect_selected_crc_data(
    count_path: str | None = None,
    metadata_path: str | None = None,
) -> dict:
    """Count matrix와 paired metadata를 점검함."""
    counts, metadata, count_file, metadata_file = _load_and_validate_crc_data(
        count_path, metadata_path
    )
    return {
        "genes": len(counts),
        "samples": counts.shape[1],
        "patients": metadata["patient"].nunique(),
        "conditions": metadata["condition"].value_counts().to_dict(),
        "paired": True,
        "count_path": str(count_file),
        "metadata_path": str(metadata_file),
    }


# ============================================================
# Tool 2. paired PyDESeq2 실행
# ============================================================
@tool
def run_selected_crc_deseq2(
    count_path: str | None = None,
    metadata_path: str | None = None,
    output_dir: str | None = None,
    min_total_count: int = 10,
    force_recompute: bool = False,
) -> dict:
    """기본 또는 업로드된 데이터로 paired DESeq2를 실행함."""
    counts, metadata, count_file, metadata_file = _load_and_validate_crc_data(
        count_path, metadata_path
    )
    result_dir = _resolve_result_dir(output_dir)
    stats_path, _, _ = _result_paths(result_dir)

    if stats_path.is_file() and not force_recompute:
        return {
            "reused": True,
            "design": "~patient + condition",
            "contrast": "tumor_vs_normal",
            "output_csv": str(stats_path),
        }

    metadata = metadata.set_index("sample")
    counts = counts.loc[counts.sum(axis=1) >= min_total_count].T.loc[metadata.index].astype(int)
    metadata["patient"] = metadata["patient"].astype(str)
    metadata["condition"] = pd.Categorical(
        metadata["condition"], categories=["normal", "tumor"]
    )

    dds = DeseqDataSet(
        counts=counts,
        metadata=metadata,
        design="~patient + condition",
        refit_cooks=True,
        n_cpus=1,
        quiet=True,
    )
    dds.deseq2()
    stats = DeseqStats(
        dds,
        contrast=["condition", "tumor", "normal"],
        alpha=0.05,
        n_cpus=1,
        quiet=True,
    )
    stats.summary()
    result = stats.results_df.copy()
    result.index.name = "gene"
    result.to_csv(stats_path)

    return {
        "reused": False,
        "design": "~patient + condition",
        "contrast": "tumor_vs_normal",
        "genes_tested": len(result),
        "count_path": str(count_file),
        "metadata_path": str(metadata_file),
        "output_csv": str(stats_path),
    }


# ============================================================
# Tool 3. DEG 선별 및 Volcano plot 생성
# ============================================================
@tool
def summarize_selected_crc_results(
    stats_path: str | None = None,
    output_dir: str | None = None,
    padj_threshold: float = 0.05,
    lfc_threshold: float = 1.0,
) -> dict:
    """전체 통계에서 DEG를 선별하고 Volcano plot을 생성함."""
    result_dir = _resolve_result_dir(output_dir)
    default_stats, deg_path, volcano_path = _result_paths(result_dir)
    stats_file = Path(stats_path).expanduser().resolve() if stats_path else default_stats
    if not stats_file.is_file() or not stats_file.is_relative_to(OUTPUT_DIR.resolve()):
        raise FileNotFoundError(f"유효한 전체 통계 파일이 없음: {stats_file}")

    df = pd.read_csv(stats_file)
    sig = (
        df["padj"].notna()
        & (df["padj"] < padj_threshold)
        & (df["log2FoldChange"].abs() > lfc_threshold)
    )
    df.loc[sig].to_csv(deg_path, index=False)
    y = -np.log10(df["padj"].clip(lower=np.finfo(float).tiny))
    colors = np.where(
        sig & (df["log2FoldChange"] > 0),
        "#d62728",
        np.where(sig & (df["log2FoldChange"] < 0), "#1f77b4", "#bdbdbd"),
    )
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(df["log2FoldChange"], y, c=colors, s=8, alpha=0.7)
    ax.axvline(lfc_threshold, ls="--", c="black", lw=0.8)
    ax.axvline(-lfc_threshold, ls="--", c="black", lw=0.8)
    ax.axhline(-np.log10(padj_threshold), ls="--", c="black", lw=0.8)
    ax.set(
        xlabel="log2FC (tumor / normal)",
        ylabel="-log10(padj)",
        title="Paired colorectal tumor vs normal",
    )
    fig.tight_layout()
    fig.savefig(volcano_path, dpi=160)
    plt.close(fig)

    return {
        "total_DEGs": int(sig.sum()),
        "up": int((sig & (df["log2FoldChange"] > 0)).sum()),
        "down": int((sig & (df["log2FoldChange"] < 0)).sum()),
        "deg_csv": str(deg_path),
        "volcano_png": str(volcano_path),
    }


# ============================================================
# Tool 4. 유의한 유전자 중 Top N 추출
# ============================================================
@tool
def get_top_differentially_expressed_genes(
    stats_path: str | None = None,
    top_n: int = 10,
) -> list[dict]:
    """유의한 유전자를 padj와 절대 LFC 기준으로 정렬하여 반환함."""
    stats_file = Path(stats_path).expanduser().resolve() if stats_path else STATS_PATH.resolve()
    if not stats_file.is_file() or not stats_file.is_relative_to(OUTPUT_DIR.resolve()):
        raise FileNotFoundError(f"유효한 전체 통계 파일이 없음: {stats_file}")
    df = pd.read_csv(stats_file).dropna(subset=["padj", "log2FoldChange"])
    ranked = (
        df.loc[df["padj"] < 0.05]
        .assign(abs_lfc=lambda x: x["log2FoldChange"].abs())
        .sort_values(["padj", "abs_lfc"], ascending=[True, False])
        .head(top_n)
    )
    return ranked[["gene", "log2FoldChange", "stat", "padj"]].to_dict("records")


## Agent에 Tool 연결해보기

In [ ]:
# Tool 모음

ANALYSIS_TOOLS = [inspect_selected_crc_data, run_selected_crc_deseq2,
                  summarize_selected_crc_results, get_top_differentially_expressed_genes]

In [ ]:
from langchain.agents import create_agent

##############################################################################
## 연습 - 분석 Tool을 Agent에 연결하세요.
# 힌트 - 앞에서 정의한 전사체 분석 Tool 묶음의 변수명을 사용하세요.

tool_agent = create_agent(
    model=llm,
    tools=____, ##
    system_prompt=SYSTEM_PROMPT,
    name="crc_tool_agent",
)
##############################################################################


In [ ]:
## 실행

tool_query = """정해진 GSE95132 데이터를 점검하고 gene, sample, 환자 수, normal/tumor 표본 수와 모든 환자가 paired인지 알려줘."""

tool_result = run_with_compact_log(
    tool_agent, tool_query, "biml-v4-tool"
)

In [ ]:
##############################################################################
## 연습 - 같은 Tool Agent에 원하는 질문을 한 번 더 입력하세요.
# 힌트 - 데이터 구조 확인처럼 제공된 Tool로 답할 수 있는 질문이 적합합니다.

my_tool_query = """
여기에 원하는 질문을 작성하세요.
""".strip()
##############################################################################

my_tool_result = run_with_compact_log(
    tool_agent,
    my_tool_query,
    "biml-v4-tool",
)


## Agent에 Short-term Memory 연결해보기

> **`thread`는 하나의 연속된 대화 session을 구분하는 단위로 각 thread는 `thread_id`라는 고유한 이름으로 식별함**

> **`InMemorySaver`는 thread별 Agent state와 message history를 메모리에 저장하는 Checkpointer임**

- 동일한 `thread_id`를 사용하면 이전 대화와 Tool observation을 복원할 수 있음
- 다른 `thread_id`를 사용하면 별개의 대화로 처리함
- 단, 메모리에만 저장되므로 커널을 종료하면 기록이 사라짐

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_checkpointer = InMemorySaver()

##############################################################################
## 연습 - Agent에 short-term memory를 연결하세요.
# 힌트 - create_agent의 checkpointer parameter를 사용하세요.

memory_agent = create_agent(
    model=llm,
    tools=ANALYSIS_TOOLS,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=____, ##
    name="crc_memory_agent",
)
##############################################################################


In [ ]:
## thread 지정

MEMORY_THREAD = "biml-v4-memory-1"

In [ ]:
## 첫 요청에서 데이터를 점검한 뒤, 두 번째 요청에서는 accession이나 표본 수를 반복하지 않고 “방금 데이터”라고 질문함.
## Agent가 같은 대상을 이해하는지 확인함.

memory_first = run_with_compact_log(
    memory_agent,
    "정해진 GSE95132 데이터를 점검하고 환자 수와 조건별 표본 수를 알려줘.",       # query 1
    MEMORY_THREAD,
)

print("\n","-------"*20, "\n")

memory_followup = run_with_compact_log(
    memory_agent,
    "방금 확인한 데이터가 paired라고 판단한 근거와 patient 보정이 필요한 이유를 설명해줘.",       # query 2
    MEMORY_THREAD,
)

In [ ]:
##############################################################################
## 연습 - 같은 thread의 대화 내용을 활용하는 후속 질문을 작성하세요.
# 힌트 - accession을 반복하지 않고 “방금 데이터”처럼 앞선 대상을 가리켜 보세요.

my_memory_query = """
여기에 원하는 후속 질문을 작성하세요.
""".strip()
##############################################################################

my_memory_result = run_with_compact_log(
    memory_agent,
    my_memory_query,
    MEMORY_THREAD,
)


## Agent에 Planning 연결해보기

> **Middleware는 Agent의 기본 Model–Tool 실행 과정 사이에 추가 기능을 삽입하는 구성요소임**

- Agent 자체를 새로 구현하지 않고도 Planning, 대화 요약, 재시도, 사용자 승인 등의 기능을 추가할 수 있음
- Planning은 복합 요청을 여러 작업으로 나누고 수행 순서와 진행 상태를 관리하는 과정으로 `TodoListMiddleware`를 연결하여 Todo 기반 Planning 기능을 추가함

> **LangChain의 `TodoListMiddleware`가 제공하는 `write_todos` Tool은 작업을 다음 상태로 관리함**

- `pending`: 아직 시작하지 않은 작업
- `in_progress`: 현재 수행 중인 작업
- `completed`: 완료된 작업

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

##############################################################################
## 연습 - 복합 작업을 Todo로 계획하는 Middleware를 연결하세요.
# 힌트 - middleware는 Middleware instance의 list를 받습니다.

planned_agent = create_agent(
    model=llm,
    tools=ANALYSIS_TOOLS,
    system_prompt=SYSTEM_PROMPT,
    middleware=____, ##
    checkpointer=memory_checkpointer,
    name="crc_planned_agent",
)
##############################################################################


In [ ]:
## Planning 확인을 위해 데이터 점검, 통계 분석, DEG/그림, Top gene, 보고서의 다섯 작업을 한 번에 요청함.

planning_query = """
정해진 GSE95132 paired 대장암 데이터로 다음 작업을 수행해줘.
1. gene, sample, 환자 수와 paired 구조를 점검해.
2. 기존 전체 통계가 있으면 재사용하고, 없으면 ~patient + condition으로 DESeq2를 실행해.
3. FDR<0.05 및 |log2FC|>1 DEG와 volcano plot을 생성해.
4. 유의한 유전자 중 padj가 작은 순서로 top 10 gene을 제시해.
5. 방법, 주요 결과, 생성 파일과 한계를 간결하게 정리해.
각 단계가 끝날 때 Todo 상태를 즉시 갱신해줘.
""".strip()

planning_result = run_with_compact_log(
    planned_agent, planning_query, "biml-v4-planning"
)

In [ ]:
##############################################################################
## 연습 - Planning이 필요한 새로운 복합 요청을 작성하세요.
# 힌트 - 서로 의존하는 작업을 3개 이상 포함하면 Todo 변화를 관찰하기 좋습니다.

my_planning_query = """
여기에 원하는 복합 분석 요청을 작성하세요.
""".strip()
##############################################################################

my_planning_result = run_with_compact_log(
    planned_agent,
    my_planning_query,
    "biml-v4-planning",
)



## RAG를 활용한 GSEApy 문서 검색

> **RAG(Retrieval-Augmented Generation)는 외부 문서에서 질문과 관련된 정보를 검색하고, 검색 결과를 LLM의 context에 추가하여 답변 생성을 돕는 방식임**

<center>
<img src="https://drive.google.com/uc?id=1AWV89Qbd-nVUwjQd9TXiCM3M4m0oDZz-" widht='400'
height='300'/><br>
[출처] <랭체인LangChain 노트> - LangChain 한국어 튜토리얼
</center>
<br>

- RAG의 검색 방식에는 vector search, hybrid search, knowledge graph search 등이 있음
- 문서 분할, 검색 결과 재정렬, query 변환 방법도 목적에 따라 다양하게 구성할 수 있음
- **이번 실습에서 활용할 방식은 아래와 같음 (전통적인 vector-based RAG)**
  - 문서를 chunk로 분할한 뒤 embedding vector로 변환
  - vector similarity search를 활용한 chunk 검색

---
**[Chunk를 저장할 Vector DB]**
- **FAISS**는 embedding vector를 저장하고 유사한 vector를 빠르게 탐색하는 vector store임
- 이번 실습에서는 LangChain의 FAISS Vector Store wrapper를 통해 문서 vector와 metadata를 함께 관리함

| 활용 클래스 | 역할 |
  |---|---|
  | `TextLoader` | 텍스트 파일을 LangChain `Document`로 변환함 |
  | `RecursiveCharacterTextSplitter` | 문서를 검색 단위인 chunk로 분할함 |
  | `HuggingFaceEmbeddings` | 문서 chunk와 query를 embedding vector로 변환함 |
  | `FAISS` | vector와 metadata를 저장하고 유사도 검색을 수행함 |

---

**[실행 방식]**

> GSEApy 튜토리얼 문서
→ chunk 분할
→ embedding 생성
→ FAISS vector store에 저장
→ query embedding 생성
→ 유사한 chunk 검색
→ 검색 결과를 Agent context에 추가

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = []
for path in sorted(KNOWLEDGE_BASE_DIR.glob("*.txt")):
    documents.extend(TextLoader(str(path), encoding="utf-8").load())

chunks = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=120,
).split_documents(documents)

# embedding 모델은 HuggingFace를 통해 간단하게 활용 가능한 모델 사용
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)
retriever = FAISS.from_documents(chunks, embedding).as_retriever(search_kwargs={"k": 3})

##############################################################################
## 연습 - 아래 검색 함수를 LangChain Tool로 변환하세요.
# 힌트 - 앞에서 사용한 decorator를 함수 바로 위에 작성하세요.

@____ ##
def search_gseapy_documentation(query: str) -> str:
    """Search local GSEApy references for grounded prerank usage."""
    found = retriever.invoke(query)
    return "\n\n".join(
        f"SOURCE={Path(doc.metadata.get('source', 'unknown')).name}\n{doc.page_content}"
        for doc in found
    )
##############################################################################


## `create_agent()`로 Subagent를 직접 연결하기

<center>
<img src="https://drive.google.com/uc?id=1aPwECOMBrjYOunmPgmnfoaX_mFZXmW_2" widht='400'
height='300'/><br>
[출처] LangGraph: Multi-Agent Workflows
</center>
<br>


**`create_agent()`을 활용해서 Multi-Agent를 만들 수 있으나 부모–자식 위임 구조 등의 관계를 직접 만들어야 함**

<br>

**[이번 실습에서의 Main Agent - Sub Agent 구현 방식]**
1. Analysis Agent와 Documentation Agent를 각각 생성함
2. 각 Agent의 `invoke()`를 호출하는 wrapper Tool을 작성함
3. 자식의 입력·thread·오류·최종 text 반환 방식을 wrapper에서 관리함
4. wrapper Tool을 Main Agent에 다시 연결함

In [ ]:
# ============================================================
# 1. 전사체 분석 전문 Agent의 역할과 행동 원칙
# ============================================================

ANALYSIS_PROMPT = """
당신은 paired bulk RNA-seq 분석을 담당하는 전문 Analysis Agent입니다.

다음 원칙을 따르세요.
1. 위임받은 범위만 수행하고 확인된 파일과 Tool 결과만 보고하세요.
2. 위임 내용에 count_path, metadata_path, output_dir가 있으면 기본 경로보다 우선하고 모든 관련 Tool에 그대로 전달하세요.
3. GSE95132 DEG 분석은 ~patient + condition, log2(tumor / normal)을 사용하세요.
4. 동일한 입력·설계·parameter로 생성된 유효한 결과가 있으면 재사용하고 불필요하게 다시 계산하지 마세요.
5. GSEA는 GSEApy prerank로 수행하며 유의한 DEG만 자르지 말고, 전체 PyDESeq2 결과에서 결측치가 없는 Wald stat을 gene별 ranking 값으로 사용하세요.
6. GSEA 입력은 stat 내림차순으로 정렬하고 gene 중복과 결측치를 확인하세요. gene library를 임의로 변경하지 마세요.
7. 사용자가 별도로 지정하지 않으면 Human, GO_Biological_Process_2025, permutation_num=100, seed=6, min_size=15, max_size=500, threads=1을 사용하세요.
8. GSEA term은 FDR q-value < 0.05만 유의하다고 판정하고, 표와 Dot Plot의 term은 |NES| 내림차순으로 선택하세요.
9. Dot Plot은 x축 NES, 색상 -log10(FDR q-value)로 표시하고, |NES| 기준 상위 20개와 figsize=(14, 10)을 유지하세요.
10. 사용자가 지정한 분석법·library·threshold·seed가 있으면 해당 값을 우선하고, 적용한 parameter를 결과에 명시하세요.
11. 요청하지 않은 분석이나 파일은 만들지 말고, 작업이 충족되면 종료하세요.
""".strip()


# ============================================================
# 2. GSEApy 문서 검색 전문 Agent의 역할과 행동 원칙
# ============================================================

DOCUMENTATION_PROMPT = """
당신은 GSEApy 사용법 조사를 담당하는 전문 Documentation Agent입니다.

다음 원칙을 따르세요.
1. Main Agent가 위임한 질문만 로컬 문서 검색 Tool로 조사하세요.
2. prerank 입력 형식, parameter, 결과 column 및 Dot Plot 사용법 중 요청에 필요한 내용만 확인하세요.
3. 검색한 문서에서 확인된 내용만 답하고 source 파일명을 제시하세요.
4. 필요한 근거를 확보하면 검색을 중단하고, 문서에 없는 내용은 추측하지 마세요.
5. 분석 실행, 데이터 탐색 및 결과 파일 생성은 수행하지 마세요.
""".strip()


In [ ]:
# ============================================================
# 3. 두 전문 Agent 생성
# ============================================================

##############################################################################
## 연습 - 각 전문 Agent에 알맞은 Tool을 연결하세요.
# 힌트 - Analysis Agent는 분석 Tool 묶음, Documentation Agent는 RAG 검색 Tool만 사용합니다.
# 힌트2 - 앞에서 정의했던 함수들: ANALYSIS_TOOLS, search_gseapy_documentation

analysis_agent = create_agent(
    model=make_llm(),
    tools=____, ##
    system_prompt=ANALYSIS_PROMPT,
    name="manual_analysis_subagent",
)

documentation_agent = create_agent(
    model=make_llm(),
    tools=____, ##
    system_prompt=DOCUMENTATION_PROMPT,
    name="manual_documentation_subagent",
)
##############################################################################


In [ ]:
# ============================================================
# 4. 전문 Agent를 Main Agent가 호출할 수 있는 Tool로 변환
# ============================================================

##############################################################################
## 연습 - 두 wrapper 함수를 Main Agent가 호출할 수 있는 Tool로 만드세요.
# 힌트 - RAG 검색 함수에 적용한 decorator와 같습니다.

@____ ##
def ask_analysis_agent(task: str) -> str:
    """
    전사체 분석 작업을 Analysis Agent에 위임하고
    해당 Agent의 최종 text 답변을 반환함.
    """
    result = analysis_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": task,
                }
            ]
        },
        config={
            "configurable": {
                # 위임 작업마다 독립적인 대화 상태를 사용함
                "thread_id": f"analysis-{time.time_ns()}",
            }
        },
    )

    return text_only(result["messages"][-1].content)

# ============================================================
# 6. Documentation Agent를 Main Agent가 호출할 수 있는 Tool로 변환
# ============================================================

@____ ##
def ask_documentation_agent(question: str) -> str:
    """
    GSEApy 관련 질문을 Documentation Agent에 위임하고
    문서 근거가 포함된 최종 text 답변을 반환함.
    """
    result = documentation_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        },
        config={
            "configurable": {
                # 문서 조사마다 독립적인 대화 상태를 사용함
                "thread_id": f"docs-{time.time_ns()}",
            }
        },
    )

    return text_only(result["messages"][-1].content)
##############################################################################


In [ ]:
# ============================================================
# 7. Main Agent의 역할과 Subagent 위임 순서 정의
# ============================================================

MAIN_AGENT_PROMPT = f"""
당신은 전사체 분석 작업을 계획·위임하고 결과를 통합하는 Main Agent입니다.

고정 경로:
- 입력 데이터: {DATA_DIR}
- 전체 DESeq2 통계: {STATS_PATH}
- 분석 결과 저장: {OUTPUT_DIR}
- GSEApy 문서: {KNOWLEDGE_BASE_DIR}

작업 원칙:
1. 같은 thread의 대화와 기존 산출물을 먼저 고려하고, 동일 조건의 유효한 결과가 있으면 재사용하세요.
2. 사용자 요청에 [현재 분석 입력]이 포함되면 해당 count_path, metadata_path, output_dir를 Analysis Agent의 task에 명시하세요.
3. 데이터 점검과 분석은 Analysis Agent에 위임하고, GSEA 요청은 Documentation Agent의 근거를 먼저 받은 뒤 Analysis Agent에 전달하세요.
4. GSEA는 기본적으로 전체 PyDESeq2 Wald stat 기반 GSEApy prerank로 해석하세요. 유의한 DEG 목록만 사용하는 ORA와 혼동하지 마세요.
5. 사용자가 별도로 지정하지 않으면 Human, GO_Biological_Process_2025, permutation_num=100, seed=6, min_size=15, max_size=500, threads=1, FDR q-value < 0.05를 사용하세요.
6. GSEA gene ranking은 결측치가 없는 전체 stat 내림차순이며, 유의한 term과 Dot Plot은 |NES| 기준 상위 20개를 사용하세요.
7. 결과를 바꾸는 핵심 조건이 모호하고 기본값을 적용할 수 없을 때만 한 가지 확인 질문을 하고, 답을 받기 전에는 Tool이나 Subagent를 호출하지 마세요.
8. 데이터 스크리닝만 요청받은 경우 결과를 보고한 뒤 다음에 원하는 분석을 한 문장으로 물어보세요.
9. 단순 질문에는 위 경로와 대화 context로 답하고, 복합 작업에만 최소한의 Todo를 사용하세요.
10. 현재 요청에서 생성하거나 명시적으로 확인한 파일만 보고하고, 결과 파일은 {OUTPUT_DIR} 아래에 저장하세요.
11. Subagent 결과가 요청을 충족하면 불필요한 재검색·재분석을 하지 말고 한국어로 간결하게 답하세요.
""".strip()


# ============================================================
# 8. 두 전문 Agent를 조정하는 Main Agent 생성
# ============================================================

##############################################################################
## 연습 - 앞에서 학습한 model, prompt, planning, memory를 이용해 Main Agent를 완성하세요.
# 힌트 - tools에는 이미 두 wrapper Tool이 연결되어 있습니다.
# 힌트2 - InMemorySaver(), TodoListMiddleware()

manual_multi_agent = create_agent(
    model=____, ##
    tools=[ask_analysis_agent, ask_documentation_agent],
    system_prompt=____, ##
    middleware=____, ##
    checkpointer=____, ##
    name="manual_multi_agent",
)
##############################################################################


### 직접 구현한 Multi-Agent 실행

- Main Agent가 Documentation wrapper와 Analysis wrapper를 올바른 순서로 호출하는지 실시간 로그에서 확인함

In [ ]:
manual_subagent_query = f"""
로컬 문서에서 Wald statistic 기반 gp.prerank 입력 형식을 먼저 조사해줘.
이후 그 근거를 분석 Agent에 전달하여 {STATS_PATH}로 GO_Biological_Process_2025에 대한 GSEA를 수행할 재현 가능한 절차를 작성하게 해
아직 분석 코드는 실행하지 말고 두 전문 Agent의 결과를 비교·정리해서 말해줘.
""".strip()

manual_subagent_result = run_with_compact_log(
    manual_multi_agent,
    manual_subagent_query,
    "biml-v4-manual-subagents",
)

In [ ]:
##############################################################################
## 연습 - 같은 Main Agent와 thread에 새로운 위임 요청을 작성하세요.
# 힌트 - Documentation Agent와 Analysis Agent의 역할이 모두 필요한 질문을 시도해 보세요.

my_manual_multi_agent_query = """
여기에 원하는 Multi-Agent 질문을 작성하세요.
""".strip()
##############################################################################

my_manual_multi_agent_result = run_with_compact_log(
    manual_multi_agent,
    my_manual_multi_agent_query,
    "biml-v4-manual-subagents",
)


# Part 2. Deep Agent Harness로 동일 구조 단순화

<table>
<tr>
<td align="center">
<img src="https://drive.google.com/uc?id=1538WWWtWLpN7NYVhsw_arNrHbkEG-X20" width='400' height='300'/><br>
[출처] LangChain Deep Agents
</td>
<td align="center">
<img src="https://drive.google.com/uc?id=132IZF3kKy-o4rbqAzbUkV1cD38sKhVNA" width='400' height='280'/><br>
[출처] Building Multi-Agent Applications with Deep Agents
</td>
</tr>
</table>

<br>

## [비교하기 1] — 기본 파일 작업과 코드 실행

- Part 1의 create_agent()는 등록된 Custom Tool만 호출할 수 있음
  - 즉, 파일 탐색·읽기·저장·코드 실행 기능이 필요하면 각각의 Tool을 직접 작성해야 함
- **기본 Harness가 탑재된 Deep Agent에서는 실행 가능한 Backend를 연결하면 아래와 같은 "파일 및 실행 Tool"이 기본으로 제공됨**
  - ls: 디렉터리 내용 확인
  - glob: pattern과 일치하는 파일 검색
  - grep: 파일 내부의 문자열 검색
  - read_file: 파일 내용 읽기
  - write_file: 새 파일 작성
  - edit_file: 기존 파일 수정
  - execute: shell 명령 또는 Python script 실행
- Deep Agent 사용 시 tools=[]로 Custom Tool을 전달하지 않아도 기존 통계 CSV를 탐색하고, Python 코드를 작성·실행하여 새로운 결과 파일을 생성할 수 있음

---

- Backend는 Deep Agent가 파일을 읽고 쓰거나 명령을 실행할 환경을 정의하는 구성요소임
  - 이번 실습에서 사용하는 `LocalShellBackend`는 현재 컴퓨터의 로컬 파일 시스템과 shell 명령을 Deep Agent의 작업 환경으로 연결함

```python
LocalShellBackend(
    root_dir=str(PROJECT_DIR),  # Agent가 파일 작업을 수행할 기준 디렉터리
    virtual_mode=False,         # 파일 경로를 가상 경로로 변환할지 여부
    timeout=180,                # 한 번의 shell 명령에 허용되는 최대 실행 시간
    inherit_env=True,           # 현재 Python 환경변수를 shell 실행 환경에 전달할지 여부
)
```

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend

## 아래 실습에서는 Part 1의 `get_top_differentially_expressed_genes`를 제공하지 않음
## Deep Agent가 기본 파일/실행 Tool만으로 Top 10 CSV를 생성하는지 확인함


# ============================================================
# 1. 파일 작업 Deep Agent의 역할과 행동 원칙
# ============================================================

FILESYSTEM_DEEP_AGENT_PROMPT = """
당신은 BIML 프로젝트 내부의 파일 분석을 담당하는 Deep Agent입니다.

다음 원칙을 따르세요.
1. 작업을 시작하기 전에 실제 파일을 탐색하고 존재 여부를 확인하세요.
2. 표 형식의 데이터 계산에는 Python을 사용하세요.
3. 원본 데이터와 기존 전체 통계 파일을 덮어쓰지 마세요.
4. 새 결과는 지정된 출력 디렉터리에 별도 파일로 저장하세요.
5. 결과 파일을 생성한 뒤 다시 읽어 내용과 경로를 검증하세요.
6. 실제로 확인하거나 생성한 파일만 보고하세요.
7. BIML 프로젝트 디렉터리 밖의 파일은 읽거나 수정하지 마세요.
""".strip()


# ============================================================
# 2. Deep Agent가 사용할 로컬 작업 환경 구성
# ============================================================

##############################################################################
## 연습 - Deep Agent가 파일을 읽고 코드를 실행할 작업 환경을 구성하세요.
# 힌트 - root_dir은 프로젝트 경로이며 절대경로를 그대로 사용합니다.

backend = LocalShellBackend(
    root_dir=____, ##
    virtual_mode=____, ##
    timeout=180,
    inherit_env=True,
)
##############################################################################

##############################################################################
## 연습 - 기본 harness와 memory를 갖춘 파일 작업 Deep Agent를 완성하세요.
# 힌트 - Custom Tool 없이 backend 기능을 사용하며 checkpointer에는 InMemorySaver를 연결합니다.

filesystem_deep_agent = create_deep_agent(
    model=make_llm(),
    tools=[],
    system_prompt=FILESYSTEM_DEEP_AGENT_PROMPT,
    backend=____, ##
    checkpointer=____, ##
    name="crc_filesystem_deep_agent",
)
##############################################################################


In [ ]:
filesystem_query = f"""
{OUTPUT_DIR}에서 GSE95132 전체 DESeq2 통계 CSV를 찾아 구조를 확인하세요.

다음 작업을 순서대로 수행하세요.
1. 전체 통계 CSV의 경로와 column을 확인하세요.
2. adjusted p-value가 유효하고 padj < 0.05인 유전자만 남기세요.
3. padj 오름차순으로 정렬하고, padj가 같으면
    |log2FoldChange| 내림차순으로 정렬하세요.
4. 상위 10개 유전자를 선택하세요.
5. 결과를 다음 파일에 저장하세요.
    {OUTPUT_DIR / "deep_agent_top10_genes.csv"}
6. 생성한 CSV를 다시 읽어 10개 유전자가 저장되었는지 검증하세요.
7. 사용한 입력 파일, 정렬 기준, Top 10 결과와 생성 파일을 보고하세요.
""".strip()


filesystem_result = run_with_compact_log(
    filesystem_deep_agent,
    filesystem_query,
    thread_id="biml-v4-deep-filesystem",
    subgraphs=True,
)

In [ ]:
##############################################################################
## 연습 - 같은 Deep Agent와 thread에 파일 작업을 한 번 더 요청하세요.
# 힌트 - 앞에서 만든 파일을 “방금 결과”라고 지칭하여 memory도 함께 확인할 수 있습니다.

my_filesystem_query = """
여기에 원하는 파일 분석 요청을 작성하세요.
""".strip()
##############################################################################

my_filesystem_result = run_with_compact_log(
    filesystem_deep_agent,
    my_filesystem_query,
    thread_id="biml-v4-deep-filesystem",
    subgraphs=True,
)


### [추가] Deep Agent의 기본 Planning 확인

앞의 실행 결과에서 볼 수 있듯이 Deep Agent는 사용자가 `TodoListMiddleware`를 직접 전달하지 않았는데도 복합 요청을 Todo로 분해하고 진행 상태를 갱신함

이는 `create_deep_agent()`가 내부의 기본 middleware 구성에 `TodoListMiddleware()`를 미리 포함하기 때문임

따라서 Deep Agent는 `write_todos` Tool을 기본적으로 사용할 수 있으며, 모델은 복합 작업이 필요하다고 판단할 때 Todo를 작성하고 상태를 갱신할 수 있음

반면 Part 1의 `create_agent()`에서는 Todo 기반 Planning을 사용하려면 다음과 같이 `TodoListMiddleware()`를 직접 연결해야 함

```python
create_agent(
    ...,
    middleware=[
        TodoListMiddleware(),
    ],
)
```

> 따라서 이번 비교에서는 파일 작업뿐 아니라 Planning middleware의 기본 포함 여부도 함께 확인할 수 있음

- `create_agent()`: `TodoListMiddleware()`를 사용자가 직접 연결함
- `create_deep_agent()`: 기본 middleware stack에 `TodoListMiddleware()`가 이미 포함되어 있음

단, Deep Agent가 모든 요청에서 반드시 Todo를 작성하는 것은 아니며, 실제 `write_todos` 호출 여부는 요청의 복잡도와 모델의 판단에 따라 달라짐


## [비교하기 2] — Subagent 정의 후 바로 연결

- Part 1에서는 자식 Agent를 생성한 뒤 `ask_*_agent` wrapper Tool, thread ID, 최종 message 추출을 직접 구현함

- Deep Agent에서는 각 전문 Agent를 dictionary로 정의하고 `subagents=[...]`에 전달하는 간단한 방식으로 구현할 수 있음

- Main에는 위임용 `task` Tool이 자동 추가되며 Subagent는 독립 context에서 실행됨

- `subgraphs=True` stream을 사용하면 부모와 자식 실행 범위를 함께 관찰할 수 있음

In [ ]:
# ============================================================
# 1. 전문 Subagent 정의
# ============================================================

# 공통 dictionary key
# - name: Main Agent가 위임 대상을 식별하는 이름
# - description: Main Agent가 위임 대상을 선택할 때 참고하는 설명
# - system_prompt: Subagent의 역할과 행동 원칙
# - tools: 해당 Subagent만 사용할 수 있는 Tool
# - model: Main Agent와 동일한 모델 및 공용 rate limiter

##############################################################################
## 연습 - 두 전문 Subagent의 prompt와 Tool 범위를 연결하세요.
# 힌트 - Analysis와 Documentation이 사용할 구성요소를 구분하세요.
# 힌트2 - ANALYSIS_TOOLS, search_gseapy_documentation, ...

analysis_subagent = {
    "name": "transcriptomics_analysis_agent",
    "description": (
        "paired 전사체 분석, DEG 선별, 시각화 및 "
        "문서 근거 기반 GSEA 실행을 담당함."
    ),
    "system_prompt": ____, ##
    "tools": ____, ##
    "model": make_llm(),
}

documentation_subagent = {
    "name": "gseapy_documentation_agent",
    "description": (
        "로컬 GSEApy 문서를 검색하여 preranked GSEA 사용법을 "
        "근거와 함께 반환하며 분석은 실행하지 않음."
    ),
    "system_prompt": ____, ##
    "tools": ____, ##
    "model": make_llm(),
}
##############################################################################


In [ ]:
# ============================================================
# 2. 두 전문 Subagent를 조정하는 Main Deep Agent 생성
# ============================================================

##############################################################################
## 연습 - dictionary로 정의한 두 Subagent를 Deep Agent에 직접 연결하세요.
# 힌트 - Part 1과 달리 wrapper Tool 없이 subagents parameter를 사용합니다.

deep_multi_agent = create_deep_agent(
    model=make_llm(),
    tools=[],
    system_prompt=MAIN_AGENT_PROMPT,
    subagents=____, ##
    backend=backend,
    checkpointer=InMemorySaver(),
    name="crc_deep_multi_agent",
)
##############################################################################


In [ ]:
# ============================================================
# 4. Main Deep Agent에 Multi-Agent 분석 요청 전달
# ============================================================

deep_gsea_query = f"""
{STATS_PATH}의 전체 Wald statistic을 사용하여 preranked GSEA를 수행하세요.

다음 순서로 작업하세요.
1. Documentation Subagent에 로컬 GSEApy 문서 조사를 위임하세요.
2. Wald statistic 기반 rank 입력 형식과 `gp.prerank()` 사용법을 확인하세요.
3. Documentation Subagent가 반환한 문서 근거를 Analysis Subagent에 전달하세요.
4. Analysis Subagent가 다음 gene set을 대상으로 GSEA를 실행하게 하세요.
    - GO_Biological_Process_2025
5. 다음 parameter를 사용하세요.
    - permutation_num=100
    - seed=6
6. 각 gene set의 전체 결과를 {OUTPUT_DIR}에 CSV로 저장하세요.
7. 생성한 파일을 다시 확인하세요.
8. 방법, 주요 관찰 결과, 생성 파일과 분석 한계를 한국어로 정리하세요.

Main Agent는 GSEA를 직접 실행하지 말고,
Documentation Subagent와 Analysis Subagent 사이의 조사·실행 순서를 조정하세요.
""".strip()


# ============================================================
# 5. Multi-Agent 실행 과정 확인
# ============================================================

deep_gsea_result = run_with_compact_log(
    deep_multi_agent,
    deep_gsea_query,
    thread_id="biml-v4-deep-gsea",
    subgraphs=True,
)

In [ ]:
add_deep_gsea_query = f"""
앞에서 수행한 GO_Biological_Process_2025 GSEA 결과로 Dot Plot을 생성하세요.

이미지는 {OUTPUT_DIR / "GO_Biological_Process_2025_dotplot.png"}에 저장하세요.
생성 후 파일의 존재 여부를 확인하고 최종 답변에도 절대경로를 포함하세요.
""".strip()

# ============================================================
# 5. Multi-Agent 실행 과정 확인
# ============================================================

deep_gsea_result = run_with_compact_log(
    deep_multi_agent,
    add_deep_gsea_query,
    thread_id="biml-v4-deep-gsea",
    subgraphs=True,
)

In [ ]:
## 옵션

add2_deep_gsea_query = f"""
앞에서 수행한 Dot Plot 결과가 마음에 들지 않아요, 수정해주세요.
가로의 길이가 짧아서 dot plot이 제대로 보이지 않아요. 크기를 조정해주세요.
그 외에도 스스로 결과를 한 번 확인하여 문제없이 제대로 구현됐는지 본 뒤 최종본을 저에게 보내주세요.

이미지는 이전과 동일한 방식으로 저장해주세요.
추가로 생성된 이미지의 절대경로를 stdout에 출력하는 방식으로 시각화해주세요.
""".strip()

# ============================================================
# 5. Multi-Agent 실행 과정 확인
# ============================================================

deep_gsea_result = run_with_compact_log(
    deep_multi_agent,
    add2_deep_gsea_query,
    thread_id="biml-v4-deep-gsea",
    subgraphs=True,
)

In [ ]:
##############################################################################
## 연습 - 같은 Multi-Agent와 thread에 자유로운 후속 요청을 작성하세요.
# 힌트 - “앞에서 만든 결과”처럼 이전 작업을 가리켜 memory와 위임 과정을 확인해 보세요.

my_deep_multi_agent_query = """
여기에 원하는 후속 요청을 작성하세요.
""".strip()
##############################################################################

my_deep_multi_agent_result = run_with_compact_log(
    deep_multi_agent,
    my_deep_multi_agent_query,
    thread_id="biml-v4-deep-gsea",
    subgraphs=True,
)


# Part 3. 방법론 최종 비교 및 Gradio로 웹 플랫폼 구현

## 최종 정리: create_agent()와 Deep Agent 비교

- create_agent()는 필요한 기능을 직접 선택하여 구성하는 범용 Tool-calling Agent임
- Deep Agent는 장기·복합 작업에서 자주 필요한 기능을 미리 조합한 Agent harness임

<br>

---

<br>

### 핵심 차이
- create_agent()에서도 Planning, Memory, Subagent 및 파일 기능을 구현할 수 있음
- 그러나 각 기능의 연결 방식과 실행 정책을 개발자가 직접 구성해야 함
- Deep Agent는 다음 기능을 기본 harness에 포함함
  - Todo 기반 Planning
  - 파일 탐색·읽기·쓰기
  - Shell 및 Python 코드 실행
  - 큰 Tool 결과의 filesystem offloading
  - 긴 대화의 자동 요약
  - Subagent 위임 및 독립 context 관리

> **따라서 Deep Agent의 핵심은 새로운 분석 방법을 자동으로 보장하는 것이 아니라, 장기·복합 작업에 필요한 실행 환경을 적은 구성 코드로 제공한다는 점임**

## 최종 Deep Transcriptomics Agent 재구성

아래는 Tool, GSEApy RAG, Analysis Subagent, Documentation Subagent, Memory 및 Backend를 하나로 다시 연결하여 Gradio에서 사용할 최종 Agent를 생성한 코드임
- 기본 설정

In [ ]:
# Part 1·2에서 정의한 경로, LLM, 분석 Tool, RAG, Subagent 및 Backend를 재사용함.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"기존 분석 구성 재사용: {OUTPUT_DIR}")


- Agent 제작

> **아래부터는 빈칸을 찾아서 채워주세요**


In [ ]:
# ============================================================
# 1. Gradio 결과 다운로드용 Tool 추가
# ============================================================

@tool
def package_analysis_results(
    file_paths: list[str],
    archive_name: str = "transcriptomics_results.zip",
) -> dict:
    """현재 요청의 OUTPUT_DIR 결과 파일을 ZIP으로 묶고 검증함."""
    output_root = OUTPUT_DIR.resolve()
    upload_root = (OUTPUT_DIR / "uploads").resolve()
    safe_name = Path(archive_name).name
    if not safe_name.lower().endswith(".zip"):
        safe_name += ".zip"
    safe_name = re.sub(r"[^A-Za-z0-9._-]", "_", safe_name)
    archive_path = OUTPUT_DIR / f"{Path(safe_name).stem}_{int(time.time())}.zip"

    selected_files = []
    for file_path in file_paths:
        candidate = Path(file_path).expanduser().resolve()
        if not candidate.is_relative_to(output_root):
            raise ValueError(f"OUTPUT_DIR 밖의 파일은 포함할 수 없음: {candidate}")
        if candidate.is_relative_to(upload_root):
            raise ValueError(f"업로드 원본은 결과 ZIP에 포함할 수 없음: {candidate}")
        if not candidate.is_file():
            raise FileNotFoundError(candidate)
        if candidate.suffix.lower() != ".zip":
            selected_files.append(candidate)

    selected_files = sorted(set(selected_files))
    if not selected_files:
        raise ValueError("ZIP에 포함할 유효한 분석 결과 파일이 없음.")

    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for result_path in selected_files:
            archive.write(result_path, arcname=result_path.relative_to(output_root))

    with zipfile.ZipFile(archive_path, "r") as archive:
        archived_files = archive.namelist()
        invalid_file = archive.testzip()

    if invalid_file is not None:
        archive_path.unlink(missing_ok=True)
        raise RuntimeError(f"ZIP 무결성 검사 실패: {invalid_file}")

    return {
        "archive_path": str(archive_path),
        "file_count": len(archived_files),
        "files": archived_files,
        "verified": True,
    }


# ============================================================
# 2. Gradio에서 사용할 최종 Deep Agent 생성
# ============================================================

##############################################################################
## 연습 - 지금까지 만든 Tool, prompt, Subagent, backend와 memory를 모두 연결하세요.
# 힌트 - MAIN_AGENT_PROMPT, analysis_subagent, documentation_subagent, InMemorySaver()

deep_transcriptomics_agent = create_deep_agent(
    model=____, ##
    tools=____, ##
    system_prompt=____, ##
    subagents=____, ##
    backend=____, ##
    checkpointer=____, ##
    name="biml_transcriptomics_deep_agent",
)
##############################################################################

print("Deep transcriptomics agent 생성 완료")
print("Subagents:", analysis_subagent["name"], documentation_subagent["name"])


## Gradio 기반 Agentic Web Platform 구성

- **Gradio는 Python 함수에 웹 UI를 연결하여 별도의 frontend 코드를 작성하지 않고도 대화형 웹 application을 만들 수 있게 해주는 open-source library**
- Textbox, Button, Chatbot과 같은 화면 요소를 Python으로 정의하고, 사용자의 click 또는 submit event가 발생했을 때 실행할 함수와 입출력을 연결하는 방식으로 동작함

---

### Gradio의 기본 구성

| 구성요소 | 역할 | 본 실습의 예 |
|---|---|---|
| `gr.Blocks()` | 여러 Component를 배치하는 application container | 전체 Agentic Web Platform |
| `gr.Row()`, `gr.Column()` | Component의 가로·세로 layout 구성 | Chatbot과 Sidebar 배치 |
| `gr.Textbox`, `gr.Button` | 사용자 입력과 event 발생 | 분석 요청 입력, 전송·중지 버튼 |
| `gr.File` | 로컬 파일을 application에 전달 | Count matrix와 metadata 업로드 |
| `gr.Chatbot` | user/assistant message 표시 | Agent의 요청과 최종 답변 |
| `gr.HTML` | HTML 형태의 상태 정보 표시 | Ready, Running, Error badge |
| `gr.State` | 화면에는 보이지 않는 session별 값 보관 | thread ID와 대화 기록 |
| `gr.DownloadButton` | 생성된 결과 파일 다운로드 | 분석 결과 ZIP 다운로드 |

<br>

**Gradio application은 일반적으로 다음 순서로 작성함**

```python
import gradio as gr


# 1. (예시) Component에서 받은 값을 처리할 Python 함수
def greet(name):
    return f"안녕하세요, {name}!"


# 2. Blocks 안에 화면 Component 배치
with gr.Blocks() as demo:
    name_input = gr.Textbox(label="이름")
    result_output = gr.Textbox(label="결과")
    run_button = gr.Button("실행")

    # 3. Event와 Python 함수 연결
    run_button.click(
        fn=greet,
        inputs=[name_input],
        outputs=[result_output],
    )


# 4. 웹 application 실행
demo.launch()
```

- `click()`과 `submit()`은 event listener임
- `inputs` Component의 현재 값이 `fn`의 argument로 전달되고, 함수의 반환값은 같은 순서의 `outputs` Component에 반영됨
- 여러 output을 지정했다면 함수도 그 개수와 순서에 맞게 값을 반환해야 함

---

### 선택적 분석 데이터 업로드

- Count matrix와 metadata를 함께 업로드하면 현재 thread 전용 입력으로 사용함
- 업로드하지 않으면 기본 GSE95132 예제 데이터를 사용함
- 업로드 파일은 내용 기반 식별자와 thread ID로 분리하여 서로 다른 사용자나 데이터가 섞이지 않게 함
- 분석 전에 sample, patient, condition, paired 구조와 count 값의 형식을 검증함
- 업로드 원본은 결과 ZIP에 포함하지 않고 생성된 분석 결과만 다운로드 대상으로 수집함

---

### Agent stream을 Gradio에 연결하는 방법

- 일반 함수는 작업이 모두 끝난 뒤 한 번 `return`하지만, Agent는 실행 시간이 길고 중간 상태가 계속 변함
- 따라서 본 실습의 `run_deep_agent_ui()`는 generator function으로 작성하고 `yield`를 사용함

```python
def run_agent(message):
    logs = []

    for event in agent.stream(...):
        logs.append(str(event))

        # 실행 중간에도 화면을 갱신함
        yield "\n".join(logs)
```

- `yield`가 실행될 때마다 Gradio가 Chatbot, status badge, Todo 및 실행 로그를 갱신하므로 사용자는 Agent의 진행 과정을 실시간으로 확인할 수 있음
  - Generator function은 `return`으로 결과를 한 번만 반환하고 종료하는 일반 함수와 달리, `yield`를 사용해 실행 도중의 결과를 여러 번 순차적으로 전달할 수 있는 Python 함수임
  - 이를 활용하면 Gradio는 Agent의 작업이 끝날 때까지 기다리지 않고, `run_deep_agent_ui()`가 전달하는 대화·Todo·로그를 실행 중에도 계속 갱신할 수 있음
- 본 플랫폼에서는 다음 흐름으로 연결됨

```text
선택적으로 Count matrix와 Metadata 업로드
        ↓
Textbox에 분석 요청 입력
        ↓
Button.click() 또는 Textbox.submit()
        ↓
run_deep_agent_ui() 실행
        ↓
Deep Agent stream 처리
        ↓
Chatbot · Todo · 실행 로그 · 다운로드 버튼 갱신
```

- `gr.State`에는 현재 `thread_id`와 session별 대화 기록을 저장함. 같은 thread ID를 Agent에 다시 전달하면 이전 대화를 이어갈 수 있고, `+ 새 분석` 버튼은 새로운 thread ID를 만들어 독립된 대화를 시작함
- `queue()`는 시간이 오래 걸리는 요청을 대기열로 관리하고 generator의 streaming update를 전달하는 데 사용함
- Colab에서는 다음과 같이 실행함

```python
demo.queue().launch(
    share=True,
    show_error=True,
    allowed_paths=[str(OUTPUT_DIR)],
    css=CUSTOM_CSS,
)
```

- `share=True`: 외부에서 접속 가능한 임시 URL 생성
- `show_error=True`: server error를 화면에 표시
- `allowed_paths`: Gradio가 제공하거나 다운로드할 수 있는 로컬 경로 허용
- `css`: application에 custom style 적용

참고: [Gradio Blocks와 Event Listener](https://www.gradio.app/main/guides/blocks-and-event-listeners/), [Gradio Blocks API](https://www.gradio.app/main/docs/gradio/blocks)


In [ ]:
import ast
import hashlib
import json
import re
import shutil
import time
import uuid
from pathlib import Path

import gradio as gr
from langchain_core.messages import AIMessage, ToolMessage


# ============================================================
# 1. 기본 설정
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/2026-BIML-Jeon-Agentic_AI")
OUTPUT_DIR = Path("/content/drive/MyDrive/BIML-Jeon-Output")

print("Gradio version:", gr.__version__)


# ============================================================
# 2. Thread 및 화면 표시 함수
# ============================================================

def make_thread_id() -> str:
    """새 대화를 구분할 thread ID를 생성함."""
    return f"biml-web-{uuid.uuid4()}"


def make_session_title(message: str, limit: int = 32) -> str:
    """첫 요청을 이용해 대화 제목을 생성함."""
    title = " ".join((message or "").split())
    return title if len(title) <= limit else title[:limit - 1] + "…"


def session_choices(sessions: dict) -> list[tuple[str, str]]:
    """Dropdown에 표시할 대화 제목과 thread ID를 반환함."""
    return [
        (session["title"], thread_id)
        for thread_id, session in reversed(list(sessions.items()))
    ]


def status_badge(status: str) -> str:
    """현재 실행 상태를 HTML badge로 반환함."""
    styles = {
        "Ready": ("#F3F4F6", "#374151", "● Ready"),
        "Running": ("#EFF6FF", "#2563EB", "● Running"),
        "Completed": ("#ECFDF5", "#047857", "● Completed"),
        "Stopped": ("#FFF7ED", "#C2410C", "■ Stopped"),
        "Error": ("#FEF2F2", "#DC2626", "● Error"),
    }
    background, color, label = styles[status]

    return f"""
    <span style="display:inline-block; padding:6px 12px; border-radius:999px;
                background:{background}; color:{color};
                font-size:13px; font-weight:600;">
        {label}
    </span>
    """


def render_todos(todos: list[dict]) -> str:
    """Todo 목록을 Sidebar용 HTML로 변환함."""
    if not todos:
        return """
        <div class="empty-panel">
            분석 요청을 기다리는 중임.
        </div>
        """

    symbols = {
        "pending": "○",
        "in_progress": "▶",
        "completed": "✓",
    }
    colors = {
        "pending": "#6B7280",
        "in_progress": "#2563EB",
        "completed": "#047857",
    }
    rows = []

    for todo in todos:
        state = todo.get("status", "pending")
        content = todo.get("content", "")
        background = "#EFF6FF" if state == "in_progress" else "transparent"

        rows.append(
            f"""
            <div style="display:flex; gap:9px; padding:8px 10px;
                        margin-bottom:3px; border-radius:7px;
                        background:{background}; color:{colors.get(state, "#6B7280")};
                        font-size:14px;">
                <span style="font-weight:700;">{symbols.get(state, "○")}</span>
                <span>{content}</span>
            </div>
            """
        )

    return "".join(rows)


def render_thread_info(thread_id: str) -> str:
    """선택된 thread ID를 작은 글씨로 표시함."""
    return f"""
    <div style="font-family:monospace; color:#6B7280;
                font-size:11px; padding-top:4px;">
        thread_id: {thread_id}
    </div>
    """


# ============================================================
# 3. Deep Agent stream 처리 Helper
# ============================================================

def text_only(content) -> str:
    """문자열 또는 provider content block에서 text만 추출함."""
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "\n".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        ).strip()

    return str(content)


def normalize_todos(value) -> list[dict]:
    """write_todos Tool argument를 UI 표시 형식으로 정리함."""
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            try:
                value = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return []

    if not isinstance(value, list):
        return []

    normalized = []

    for todo in value:
        if not isinstance(todo, dict):
            continue

        # 일부 모델이 -content로 잘못 반환하는 경우도 처리
        content = todo.get("content") or todo.get("-content")
        status = todo.get("status", "pending")

        if not content:
            continue

        if status not in {"pending", "in_progress", "completed"}:
            status = "pending"

        normalized.append(
            {
                "content": str(content),
                "status": status,
            }
        )

    return normalized


def parse_tool_content(content):
    """ToolMessage content를 가능한 경우 Python 객체로 변환함."""
    if isinstance(content, dict):
        return content

    if not isinstance(content, str):
        return content

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(content)
        except (ValueError, SyntaxError):
            return content


def extract_archive_path(content) -> str | None:
    """
    package_analysis_results의 Tool observation에서
    검증된 ZIP 절대경로를 추출함.
    """
    parsed = parse_tool_content(content)
    archive_path = None
    verified = False

    if isinstance(parsed, dict):
        archive_path = parsed.get("archive_path")
        verified = parsed.get("verified") is True

    # Tool 출력이 일반 문자열로 직렬화된 경우 보조 검색
    if archive_path is None:
        text = str(content)
        marker = str(OUTPUT_DIR)

        for token in text.replace('"', " ").replace("'", " ").split():
            candidate = token.rstrip(".,;:)}]")

            if candidate.startswith(marker) and candidate.endswith(".zip"):
                archive_path = candidate
                verified = True
                break

    if not archive_path or not verified:
        return None

    path = Path(archive_path).expanduser().resolve()
    output_root = OUTPUT_DIR.resolve()

    if not path.is_relative_to(output_root):
        return None

    if path.suffix.lower() != ".zip":
        return None

    if not path.is_file():
        return None

    return str(path)



RESULT_EXTENSIONS = {
    ".csv", ".tsv", ".png", ".jpg", ".jpeg", ".pdf", ".txt", ".html"
}
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}


def snapshot_output_images() -> dict[Path, tuple[int, int]]:
    """
    현재 OUTPUT_DIR 이미지의 version 정보를 기록함.

    기존 이미지를 표시하기 위한 함수가 아니라 현재 요청에서
    새로 생성되거나 수정된 이미지를 구분하기 위한 snapshot임.
    """
    output_root = OUTPUT_DIR.resolve()
    snapshot = {}

    if not output_root.is_dir():
        return snapshot

    for path in output_root.rglob("*"):
        if (
            not path.is_file()
            or path.suffix.lower() not in IMAGE_EXTENSIONS
        ):
            continue

        try:
            resolved = path.resolve()
            stat = resolved.stat()
        except (OSError, RuntimeError):
            continue

        if not resolved.is_relative_to(output_root):
            continue

        snapshot[resolved] = (
            stat.st_mtime_ns,
            stat.st_size,
        )

    return snapshot


def persist_session_inputs(
    count_upload,
    metadata_upload,
    thread_id: str,
    session: dict,
) -> dict:
    """업로드 파일을 thread별로 저장·검증하고 사용할 경로를 반환함."""
    if bool(count_upload) != bool(metadata_upload):
        raise ValueError("Count matrix와 Metadata를 모두 업로드해야 합니다.")

    if count_upload and metadata_upload:
        count_source = Path(str(count_upload)).expanduser().resolve()
        metadata_source = Path(str(metadata_upload)).expanduser().resolve()

        def file_digest(file_path: Path) -> str:
            digest = hashlib.sha256()
            with file_path.open("rb") as handle:
                for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                    digest.update(chunk)
            return digest.hexdigest()

        fingerprint = hashlib.sha256(
            (file_digest(count_source) + file_digest(metadata_source)).encode()
        ).hexdigest()[:12]
        upload_dir = (OUTPUT_DIR / "uploads" / thread_id / fingerprint).resolve()
        result_dir = (OUTPUT_DIR / "thread_results" / thread_id / fingerprint).resolve()
        upload_dir.mkdir(parents=True, exist_ok=True)
        result_dir.mkdir(parents=True, exist_ok=True)

        count_target = upload_dir / f"counts{count_source.suffix.lower()}"
        metadata_target = upload_dir / f"metadata{metadata_source.suffix.lower()}"

        if count_source != count_target:
            shutil.copy2(count_source, count_target)
        if metadata_source != metadata_target:
            shutil.copy2(metadata_source, metadata_target)

        _load_and_validate_crc_data(str(count_target), str(metadata_target))
        session["input_paths"] = {
            "count_path": str(count_target),
            "metadata_path": str(metadata_target),
            "output_dir": str(result_dir),
            "source": "uploaded",
        }

    return session.get("input_paths") or {
        "count_path": str(COUNT_PATH.resolve()),
        "metadata_path": str(METADATA_PATH.resolve()),
        "output_dir": str(OUTPUT_DIR.resolve()),
        "source": "example",
    }


def collect_output_paths(content, result_paths: set[Path]) -> None:
    """Agent/Tool 출력에서 OUTPUT_DIR 내부의 결과 파일을 수집함."""
    if isinstance(content, str):
        text = content
    else:
        text = json.dumps(content, ensure_ascii=False, default=str)

    output_root = OUTPUT_DIR.resolve()
    upload_root = (OUTPUT_DIR / "uploads").resolve()
    candidates = re.findall(r"/[^\s'\"`}\]]+", text)

    for candidate in candidates:
        candidate = candidate.rstrip(".,;:)}]")

        try:
            path = Path(candidate).expanduser().resolve()
        except (OSError, RuntimeError):
            continue

        if path.suffix.lower() not in RESULT_EXTENSIONS:
            continue
        if not path.is_file() or not path.is_relative_to(output_root):
            continue
        if path.is_relative_to(upload_root):
            continue

        result_paths.add(path)


def ensure_result_archive(result_paths: set[Path], thread_id: str) -> str | None:
    """분석 산출물이 있으면 Deep Agent의 ZIP Tool을 확정적으로 실행함."""
    files = sorted(
        str(path) for path in result_paths
        if path.is_file() and path.suffix.lower() != ".zip"
    )
    if not files:
        return None

    result = package_analysis_results.invoke(
        {
            "file_paths": files,
            "archive_name": f"{thread_id}_results.zip",
        }
    )
    return extract_archive_path(result)

def friendly_ui_error(exc: Exception) -> str:
    """API 및 실행 오류를 짧은 메시지로 변환함."""
    if "_friendly_error" in globals():
        return _friendly_error(exc)

    text = str(exc)

    if "RequestsPerDay" in text or "PerDay" in text:
        return "Gemini 일일 요청 한도에 도달함."

    if (
        "RequestsPerMinute" in text
        or "RESOURCE_EXHAUSTED" in text
        or "429" in text
    ):
        return "Gemini 요청 속도 한도에 도달함. 잠시 후 다시 시도하세요."

    return f"{type(exc).__name__}: {text}"



TOOL_LABELS = {
    "glob": "결과 파일 탐색",
    "read_file": "결과 파일 확인",
    "write_file": "결과 파일 저장",
    "execute": "분석 코드 실행",
    "inspect_selected_crc_data": "데이터 구조 점검",
    "run_selected_crc_deseq2": "paired DESeq2 분석",
    "summarize_selected_crc_results": "DEG 선별 및 Volcano plot 생성",
    "get_top_differentially_expressed_genes": "Top DEG 추출",
    "search_gseapy_documentation": "GSEApy 문서 검색",
    "package_analysis_results": "결과 ZIP 생성 및 검증",
}

AGENT_LABELS = {
    "transcriptomics_analysis_agent": "Analysis Agent",
    "gseapy_documentation_agent": "Documentation Agent",
}


def readable_tool_name(tool_name: str) -> str:
    """내부 Tool 이름을 사용자용 작업명으로 변환함."""
    return TOOL_LABELS.get(tool_name, tool_name.replace("_", " "))


def readable_agent_name(subagent_type: str | None) -> str:
    """내부 Subagent type을 사용자용 이름으로 변환함."""
    if not subagent_type:
        return "Subagent"
    return AGENT_LABELS.get(subagent_type, subagent_type.replace("_", " ").title())

# ============================================================
# 4. Deep Agent와 연결된 실제 Gradio 실행 함수
# ============================================================

def run_deep_agent_ui(message, history, thread_id, sessions, count_upload, metadata_upload):
    """
    deep_transcriptomics_agent의 Main/Subagent update stream을
    Chatbot, Todo, 실행 로그 및 ZIP 다운로드에 연결함.
    """
    message = (message or "").strip()
    history = list(history or [])
    sessions = dict(sessions or {})

    def selector_update():
        return gr.update(
            choices=session_choices(sessions),
            value=thread_id,
        )

    def disabled_download():
        return gr.update(
            value=None,
            interactive=False,
        )

    if not message:
        yield (
            history,
            status_badge("Ready"),
            render_todos([]),
            "",
            disabled_download(),
            thread_id,
            sessions,
            selector_update(),
        )
        return

    if not thread_id:
        thread_id = make_thread_id()

    if thread_id not in sessions:
        sessions[thread_id] = {
            "title": "새 분석",
            "history": [],
            "input_paths": None,
        }

    if not sessions[thread_id].get("history"):
        sessions[thread_id]["title"] = make_session_title(message)

    history.append(
        {
            "role": "user",
            "content": message,
        }
    )
    sessions[thread_id]["history"] = history

    started = time.monotonic()
    todos = []
    logs = ["00:00  사용자 요청 수신"]
    seen_logs = set(logs)

    final_answer = None
    archive_path = None
    result_paths: set[Path] = set()

    # 요청 시작 전에 존재하던 이미지는 자동 표시 대상에서 제외함.
    image_snapshot = snapshot_output_images()
    display_image_paths: set[Path] = set()
    displayed_image_versions = set()

    namespace_labels: dict[str, str] = {}
    pending_subagents: list[str] = []
    delegated_subagents: list[str] = []

    def elapsed() -> str:
        seconds = int(time.monotonic() - started)
        return f"{seconds // 60:02d}:{seconds % 60:02d}"

    def add_log(line: str):
        if line not in seen_logs:
            logs.append(line)
            seen_logs.add(line)

    def collect_changed_images():
        """
        직전 snapshot 이후 생성되거나 수정된 이미지만 수집함.

        이전 요청에서 생성됐지만 이번 요청에서 변경되지 않은 이미지는
        표시하지 않으며, 동일 경로를 덮어쓴 경우에는 새 version을 표시함.
        """
        nonlocal image_snapshot

        current_snapshot = snapshot_output_images()

        for path, version in current_snapshot.items():
            if image_snapshot.get(path) == version:
                continue

            image_version = (path, version)

            if image_version in displayed_image_versions:
                continue

            display_image_paths.add(path)
            displayed_image_versions.add(image_version)
            result_paths.add(path)

            add_log(
                f"{elapsed()}  새 시각화 확인 · {path.name}"
            )

        image_snapshot = current_snapshot

    def download_update():
        if archive_path:
            return gr.update(
                value=archive_path,
                interactive=True,
            )

        return disabled_download()

    def running_output():
        sessions[thread_id]["history"] = history

        return (
            history,
            status_badge("Running"),
            render_todos(todos),
            "\n".join(logs[-200:]),
            download_update(),
            thread_id,
            sessions,
            selector_update(),
        )

    yield running_output()

    try:
        input_paths = persist_session_inputs(
            count_upload,
            metadata_upload,
            thread_id,
            sessions[thread_id],
        )
        source_label = "업로드 데이터" if input_paths["source"] == "uploaded" else "GSE95132 예제 데이터"
        add_log(f"{elapsed()}  입력 데이터 · {source_label}")
        agent_message = f"""[현재 분석 입력]
count_path: {input_paths['count_path']}
metadata_path: {input_paths['metadata_path']}
output_dir: {input_paths['output_dir']}
Analysis Agent는 위 경로를 관련 Tool argument에 전달하세요.

[사용자 요청]
{message}"""
        yield running_output()

        stream = deep_transcriptomics_agent.stream(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": agent_message,
                    }
                ]
            },
            config={
                "configurable": {
                    "thread_id": thread_id,
                }
            },
            stream_mode="updates",
            subgraphs=True,
        )

        for event in stream:
            if isinstance(event, tuple):
                namespace, update = event
            else:
                namespace, update = (), event

            if namespace:
                namespace_key = " > ".join(str(item) for item in namespace)
                if namespace_key not in namespace_labels:
                    namespace_labels[namespace_key] = (
                        pending_subagents.pop(0)
                        if pending_subagents
                        else "Subagent"
                    )
                scope = namespace_labels[namespace_key]
            else:
                scope = "Main Agent"

            if not isinstance(update, dict):
                continue

            ui_changed = False

            for payload in update.values():
                if not isinstance(payload, dict):
                    continue

                agent_messages = payload.get("messages", [])

                for agent_message in agent_messages:
                    # ----------------------------------------
                    # Agent의 Tool 호출
                    # ----------------------------------------
                    if (
                        isinstance(agent_message, AIMessage)
                        and agent_message.tool_calls
                    ):
                        for call in agent_message.tool_calls:
                            tool_name = call.get("name", "unknown_tool")
                            arguments = call.get("args", {})

                            if tool_name == "write_todos":
                                raw_todos = (
                                    arguments.get("todos", [])
                                    if isinstance(arguments, dict)
                                    else []
                                )
                                parsed_todos = normalize_todos(raw_todos)

                                if parsed_todos:
                                    todos = parsed_todos

                                add_log(f"{elapsed()}  {scope} · 작업 계획 업데이트")

                            elif tool_name == "task":
                                subagent_type = (
                                    arguments.get("subagent_type")
                                    if isinstance(arguments, dict)
                                    else None
                                )
                                subagent_label = readable_agent_name(subagent_type)
                                pending_subagents.append(subagent_label)
                                delegated_subagents.append(subagent_label)
                                add_log(
                                    f"{elapsed()}  Main Agent → "
                                    f"{subagent_label} · 작업 위임"
                                )

                            else:
                                task_label = readable_tool_name(tool_name)
                                add_log(
                                    f"{elapsed()}  {scope} · "
                                    f"{task_label} 시작"
                                )

                            ui_changed = True

                    # ----------------------------------------
                    # Tool 실행 완료
                    # ----------------------------------------
                    elif isinstance(agent_message, ToolMessage):
                        tool_name = agent_message.name or "unknown_tool"
                        collect_output_paths(agent_message.content, result_paths)

                        # Tool 종류나 stdout 문구가 아니라 실제 파일 상태를 비교함.
                        # 현재 요청에서 생성·수정된 이미지만 표시 대상으로 수집함.
                        collect_changed_images()

                        if tool_name == "package_analysis_results":
                            detected_archive = extract_archive_path(
                                agent_message.content
                            )

                            if detected_archive:
                                archive_path = detected_archive
                                add_log(
                                    f"{elapsed()}  결과 ZIP 준비 완료"
                                )

                        if tool_name == "task":
                            subagent_label = (
                                delegated_subagents.pop(0)
                                if delegated_subagents
                                else "Subagent"
                            )
                            add_log(
                                f"{elapsed()}  {subagent_label} → "
                                "Main Agent · 결과 반환"
                            )
                        elif tool_name != "write_todos":
                            task_label = readable_tool_name(tool_name)
                            add_log(
                                f"{elapsed()}  {scope} · "
                                f"{task_label} 완료"
                            )

                        ui_changed = True

                    # ----------------------------------------
                    # Main Agent의 최종 답변
                    # ----------------------------------------
                    elif (
                        isinstance(agent_message, AIMessage)
                        and agent_message.content
                    ):
                        response_text = text_only(agent_message.content)
                        collect_output_paths(response_text, result_paths)

                        # Subagent 답변은 로그에만 반영하고,
                        # main namespace의 답변만 최종 Chatbot 답변으로 사용
                        if not namespace and response_text:
                            final_answer = response_text
                            ui_changed = True

            if ui_changed:
                yield running_output()

        if not final_answer:
            final_answer = (
                "Agent 실행은 종료되었지만 최종 답변을 받지 못했습니다. "
                "실행 로그를 확인해주세요."
            )

        # 마지막 ToolMessage 직후 비동기 저장이 끝난 이미지까지 확인함.
        collect_changed_images()

        # 모델이 ZIP Tool 호출을 빠뜨려도 분석 산출물이 있으면 동일 Tool을 실행함.
        if archive_path is None and result_paths:
            add_log(f"{elapsed()}  Main Agent · 결과 ZIP 생성 시작")
            archive_path = ensure_result_archive(result_paths, thread_id)

        history.append(
            {
                "role": "assistant",
                "content": final_answer,
            }
        )

        # 모든 기존 PNG가 아니라 이번 요청에서 생성·수정된 이미지만 표시함.
        image_paths = sorted(
            path for path in display_image_paths
            if path.is_file()
        )

        for path in image_paths:
            history.append(
                {
                    "role": "assistant",
                    "content": {
                        "path": str(path),
                        "alt_text": path.name,
                    },
                }
            )

        sessions[thread_id]["history"] = history
        add_log(f"{elapsed()}  전체 작업 완료")

        if archive_path:
            add_log(f"{elapsed()}  결과 ZIP 다운로드 준비 완료")
        else:
            add_log(f"{elapsed()}  생성된 분석 결과 ZIP 없음")

        yield (
            history,
            status_badge("Completed"),
            render_todos(todos),
            "\n".join(logs[-200:]),
            download_update(),
            thread_id,
            sessions,
            selector_update(),
        )

    except GeneratorExit:
        # Gradio Stop 버튼이 실행 event를 취소한 경우
        raise

    except Exception as exc:
        error_text = friendly_ui_error(exc)

        history.append(
            {
                "role": "assistant",
                "content": (
                    "분석 중 오류가 발생했습니다.\n\n"
                    f"`{error_text}`"
                ),
            }
        )
        sessions[thread_id]["history"] = history

        add_log(f"{elapsed()}  오류 · {error_text}")

        yield (
            history,
            status_badge("Error"),
            render_todos(todos),
            "\n".join(logs[-200:]),
            disabled_download(),
            thread_id,
            sessions,
            selector_update(),
        )


# ============================================================
# 5. Thread 제어 함수
# ============================================================

def start_new_analysis(sessions):
    """새 Thread를 생성하고 빈 대화 화면으로 전환함."""
    sessions = dict(sessions or {})
    thread_id = make_thread_id()

    sessions[thread_id] = {
        "title": "새 분석",
        "history": [],
        "input_paths": None,
    }

    return (
        [],
        status_badge("Ready"),
        render_todos([]),
        "",
        gr.update(value=None, interactive=False),
        thread_id,
        sessions,
        gr.update(
            choices=session_choices(sessions),
            value=thread_id,
        ),
        render_thread_info(thread_id),
        "",
        None,
        None,
    )


def load_session(selected_thread_id, sessions):
    """선택한 Thread의 대화 기록을 불러옴."""
    sessions = dict(sessions or {})

    if (
        not selected_thread_id
        or selected_thread_id not in sessions
    ):
        return (
            [],
            selected_thread_id,
            status_badge("Ready"),
            render_todos([]),
            "",
            "",
            None,
            None,
        )

    session = sessions[selected_thread_id]
    history = session.get("history", [])
    input_paths = session.get("input_paths") or {}
    count_value = input_paths.get("count_path") if input_paths.get("source") == "uploaded" else None
    metadata_value = input_paths.get("metadata_path") if input_paths.get("source") == "uploaded" else None

    return (
        history,
        selected_thread_id,
        status_badge("Ready"),
        render_todos([]),
        "이전 대화를 불러옴.",
        render_thread_info(selected_thread_id),
        count_value,
        metadata_value,
    )


def stop_analysis(history, thread_id):
    """현재 실행을 중지 상태로 변경함."""
    return (
        history or [],
        status_badge("Stopped"),
        "사용자가 실행을 중지함.",
        thread_id,
    )

# ============================================================
# 5. 초기 상태
# ============================================================

INITIAL_THREAD_ID = make_thread_id()
INITIAL_SESSIONS = {
    INITIAL_THREAD_ID: {
        "title": "새 분석",
        "history": [],
        "input_paths": None,
    }
}


# ============================================================
# 6. CSS
# ============================================================

CUSTOM_CSS = """
.gradio-container {
    width: 96vw !important;
    max-width: none !important;
    margin: 0 auto !important;
    background: #FFFFFF;
    color: #1F2937;
}

#header {
    align-items: center !important;
    border-bottom: 1px solid #E5E7EB;
    padding-bottom: 10px;
    margin-bottom: 8px;
}

#service-title h1 {
    margin: 0;
    font-size: 23px;
    color: #1F2937;
}

#service-title p {
    margin-top: 3px;
    color: #6B7280;
    font-size: 13px;
}

#history-panel {
    margin-bottom: 10px;
    border: none !important;
    background: #FFFFFF !important;
}

#main-layout {
    display: flex !important;
    flex-wrap: nowrap !important;
    align-items: stretch !important;
    gap: 18px;
}

#main-column {
    flex: 1 1 auto !important;
    min-width: 0 !important;
}

#sidebar {
    flex: 0 0 400px !important;
    width: 400px !important;
    min-width: 380px !important;
    max-width: 400px !important;
    padding: 14px;
    background: #F8FAFC;
    border: 1px solid #E5E7EB;
    border-radius: 12px;
}

#conversation {
    min-height: 650px;
    border: none !important;
    box-shadow: none !important;
    background: #FFFFFF !important;
}

#conversation > div {
    border: none !important;
    box-shadow: none !important;
}

/* Assistant 답변을 문서처럼 표시 */
#conversation .message.bot,
#conversation .message.assistant {
    width: 100% !important;
    max-width: 100% !important;
    margin: 8px 0 22px 0 !important;
    padding: 4px 8px !important;
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    color: #1F2937 !important;
}

/* User 메시지만 오른쪽 말풍선으로 표시 */
#conversation .message.user {
    width: fit-content !important;
    max-width: 72% !important;
    margin: 8px 0 14px auto !important;
    padding: 10px 14px !important;
    background: #EFF6FF !important;
    border: 1px solid #BFDBFE !important;
    border-radius: 16px 16px 4px 16px !important;
    box-shadow: none !important;
    color: #1F2937 !important;
}

.sidebar-title {
    margin: 12px 0 8px 0;
    color: #1F2937;
    font-size: 14px;
    font-weight: 700;
}

.empty-panel {
    padding: 12px 10px;
    color: #6B7280;
    font-size: 13px;
}

#execution-log textarea {
    background: #FFFFFF;
    color: #374151;
    font-family: monospace;
    font-size: 12px;
    line-height: 1.6;
}

#query-row {
    margin-top: 8px;
    padding: 6px;
    background: #FFFFFF;
    border: 1px solid #E5E7EB;
    border-radius: 16px;
}

#query-row textarea {
    border: none !important;
    box-shadow: none !important;
}

#send-button {
    background: #2563EB !important;
    color: #FFFFFF !important;
    border: none !important;
    border-radius: 12px !important;
}

#stop-button {
    color: #DC2626 !important;
    border-color: #FCA5A5 !important;
    border-radius: 12px !important;
}

#new-analysis-button {
    color: #2563EB !important;
    border-color: #BFDBFE !important;
}

@media (max-width: 950px) {
    #main-layout {
        flex-wrap: wrap !important;
    }

    #sidebar {
        flex: 1 1 100% !important;
        width: 100% !important;
        min-width: 100% !important;
        max-width: none !important;
    }
}
"""


# ============================================================
# 7. Gradio UI
# ============================================================

with gr.Blocks(title="Transcriptomics Multi-Agent") as demo:
    thread_state = gr.State(INITIAL_THREAD_ID)
    sessions_state = gr.State(INITIAL_SESSIONS)

    with gr.Row(elem_id="header"):
        with gr.Column(scale=8, elem_id="service-title"):
            gr.Markdown(
                """
# Transcriptomics Multi-Agent

Deep Agent 기반 paired 전사체 분석 플랫폼
                """.strip()
            )

        with gr.Column(scale=1, min_width=120):
            status_view = gr.HTML(status_badge("Ready"))

        with gr.Column(scale=1, min_width=120):
            new_analysis_button = gr.Button(
                "+ 새 분석",
                elem_id="new-analysis-button",
            )

    # 기본적으로 접혀 있는 Thread 기록
    with gr.Accordion("대화 기록", open=False, elem_id="history-panel"):
        thread_selector = gr.Dropdown(
            choices=session_choices(INITIAL_SESSIONS),
            value=INITIAL_THREAD_ID,
            label="이전 대화 선택",
        )
        thread_info = gr.HTML(render_thread_info(INITIAL_THREAD_ID))

    with gr.Row(elem_id="main-layout"):
        # 왼쪽: 자연스러운 대화 및 보고서
        with gr.Column(scale=7, elem_id="main-column"):
            chatbot = gr.Chatbot(
                value=[],
                height=650,
                show_label=False,
                elem_id="conversation",
            )

            with gr.Row(elem_id="query-row"):
                query_input = gr.Textbox(
                    placeholder="분석을 요청하세요...",
                    lines=2,
                    max_lines=5,
                    show_label=False,
                    scale=8,
                )
                stop_button = gr.Button("■ 중지", elem_id="stop-button", scale=1)
                send_button = gr.Button(
                    "전송 ↑",
                    variant="primary",
                    elem_id="send-button",
                    scale=1,
                )

        # 오른쪽: Todo, 실행 로그, 결과 파일
        with gr.Column(scale=3, min_width=380, elem_id="sidebar"):
            gr.HTML('<div class="sidebar-title">Input Data</div>')
            with gr.Accordion("선택적 데이터 업로드", open=False):
                count_upload = gr.File(
                    label="Count matrix (.tsv 또는 .csv)",
                    file_types=[".tsv", ".csv"],
                    type="filepath",
                )
                metadata_upload = gr.File(
                    label="Metadata (.tsv 또는 .csv)",
                    file_types=[".tsv", ".csv"],
                    type="filepath",
                )
                gr.Markdown(
                    "두 파일을 모두 올리면 현재 thread에서 사용하며, "
                    "업로드하지 않으면 GSE95132 예제 데이터를 사용합니다."
                )

            gr.HTML('<div class="sidebar-title">Analysis Todo</div>')
            todo_view = gr.HTML(render_todos([]))

            gr.HTML('<div class="sidebar-title">Execution Log</div>')
            log_view = gr.Textbox(
                value="",
                lines=16,
                max_lines=16,
                interactive=False,
                show_label=False,
                elem_id="execution-log",
            )

            gr.HTML('<div class="sidebar-title">Results</div>')
            download_button = gr.DownloadButton(
                label="결과 ZIP 다운로드",
                value=None,
                interactive=False,
            )

    # 전송 버튼
    submit_event = send_button.click(
        fn=run_deep_agent_ui,
        inputs=[query_input, chatbot, thread_state, sessions_state, count_upload, metadata_upload],
        outputs=[
            chatbot,
            status_view,
            todo_view,
            log_view,
            download_button,
            thread_state,
            sessions_state,
            thread_selector,
        ],
    )

    # Enter 전송
    enter_event = query_input.submit(
        fn=run_deep_agent_ui,
        inputs=[query_input, chatbot, thread_state, sessions_state, count_upload, metadata_upload],
        outputs=[
            chatbot,
            status_view,
            todo_view,
            log_view,
            download_button,
            thread_state,
            sessions_state,
            thread_selector,
        ],
    )

    submit_event.then(fn=lambda: "", inputs=None, outputs=query_input)
    enter_event.then(fn=lambda: "", inputs=None, outputs=query_input)

    # 새 분석
    new_analysis_button.click(
        fn=start_new_analysis,
        inputs=[sessions_state],
        outputs=[
            chatbot,
            status_view,
            todo_view,
            log_view,
            download_button,
            thread_state,
            sessions_state,
            thread_selector,
            thread_info,
            query_input,
            count_upload,
            metadata_upload,
        ],
    )

    # 이전 Thread 선택
    thread_selector.change(
        fn=load_session,
        inputs=[thread_selector, sessions_state],
        outputs=[
            chatbot,
            thread_state,
            status_view,
            todo_view,
            log_view,
            thread_info,
            count_upload,
            metadata_upload,
        ],
    )

    # 실행 중지
    stop_button.click(
        fn=stop_analysis,
        inputs=[chatbot, thread_state],
        outputs=[chatbot, status_view, log_view, thread_state],
        cancels=[submit_event, enter_event],
    )


## Gradio 웹 서비스 실행

In [ ]:
demo.queue().launch(
    share=True,
    show_error=True,
    allowed_paths=[str(OUTPUT_DIR)],
    css=CUSTOM_CSS,
)